# 09 — Claude Haiku Grading Evaluator

Exact Python port of the Chrome extension grading pipeline for offline evaluation.

Replicates **every step** of `claudeVision.ts` + `cvDetectors.ts`:
1. CV detectors — Laplacian blur, glare fraction, brightness (same constants as production)
2. Image resize — longest edge ≤ 1024 px, JPEG q85
3. Prompt injection — CV measurements prepended verbatim
4. Claude Haiku call — same model, max_tokens, system/user prompts
5. Distribution normalisation

Use this to:
- Test new prompts before deploying
- Evaluate consistency across multiple images of the same card
- Build a labelled dataset for accuracy benchmarking
- Debug why a particular card got an unexpected grade

In [ ]:
# Run once, then comment out
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'anthropic', 'Pillow', 'numpy', 'opencv-python', 'requests',
    'matplotlib', 'pandas', 'tqdm'])
print('✅ Dependencies ready')

In [ ]:
import os, io, re, json, base64, textwrap, warnings
from pathlib import Path

import numpy as np
import cv2
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import anthropic
from tqdm.auto import tqdm
from IPython.display import display as ipy_display, HTML

warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi':       120,
    'axes.facecolor':   '#161b22',
    'figure.facecolor': '#0d1117',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('✅ Imports OK')

## Configuration

Set your Anthropic API key. The notebook reads `ANTHROPIC_API_KEY` from the environment, or you can paste it directly.

In [ ]:
# ── API Key ───────────────────────────────────────────────────────────────────
API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not API_KEY:
    API_KEY = input('Paste your Anthropic API key: ').strip()
assert API_KEY, 'API key required'
print(f'✅ API key loaded ({API_KEY[:8]}...)')

## CV Detectors + Side Classifier

Exact Python port of `cvDetectors.ts`. Same constants, same algorithm.

| Constant | Value | Meaning |
|---|---|---|
| `CANONICAL_W` | 384 | Resize width for CV analysis |
| `CANONICAL_H` | 544 | Resize height for CV analysis |
| `BLUR_THRESHOLD` | 40 | Laplacian variance below = blurry |
| `GLARE_THRESHOLD` | 0.05 | Fraction of pixels ≥ 245 above = significant glare |
| `BACK_HUE_MIN/MAX` | 195–220° | HSV hue band for Pokémon card back blue |
| `BACK_SAT_MIN` | 0.50 | Min saturation for back-blue pixels |
| `BACK_PIXEL_THRESHOLD` | 0.25 | >25% matching pixels → classify as back |

In [ ]:
# ── Constants — must match cvDetectors.ts ─────────────────────────────────────
CANONICAL_W     = 384
CANONICAL_H     = 544
BLUR_THRESHOLD  = 40
GLARE_THRESHOLD = 0.05

# Corner analysis
CORNER_W_PX          = 46
CORNER_H_PX          = 55
WHITE_THRESH          = 215
WHITENING_THRESHOLD   = 0.07

# Detector A — Border Irregularity
BORDER_BAND_W              = 55
BORDER_BAND_H              = 70
BORDER_GRAD_THRESHOLD      = 30
BORDER_WHITE_THRESH        = 215

# Detector B — Surface Lines
SURFACE_MARGIN_W           = 55
SURFACE_MARGIN_H           = 70
SURFACE_GRAD_THRESHOLD     = 25
SURFACE_GLARE_THRESH       = 245

# Side classifier HSV thresholds
BACK_HUE_MIN         = 195
BACK_HUE_MAX         = 220
BACK_SAT_MIN         = 0.50
BACK_VAL_MIN         = 0.25
BACK_PIXEL_THRESHOLD = 0.25

# Centering (perspective warp pipeline)
CARD_ASPECT_W  = 500   # canonical warped card width  (px)
CARD_ASPECT_H  = 700   # canonical warped card height (px)
BORDER_SCAN_W  = 50    # scan width for border measurement (px from each edge)
BORDER_SCAN_H  = 60    # scan height for border measurement


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — image quality
# ══════════════════════════════════════════════════════════════════════════════

def _laplacian_variance(gray: np.ndarray) -> float:
    lap = cv2.Laplacian(gray.astype(np.float64), cv2.CV_64F)
    return float(lap.var())


def _analyse_image_array(img_rgb: np.ndarray) -> dict:
    """Blur / glare / brightness on the canonical resized image."""
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H),
                           interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    total            = gray.size
    brightness_mean  = float(gray.mean())
    brightness_std   = float(gray.std())
    glare_count      = int(np.sum(gray >= 245))
    glare_fraction   = glare_count / total
    blur_score       = _laplacian_variance(gray)
    is_blurry        = blur_score < BLUR_THRESHOLD
    has_glare        = glare_fraction > GLARE_THRESHOLD

    cv_issues = []
    if is_blurry:
        cv_issues.append(f'blurry image (score {blur_score:.1f}, need ≥{BLUR_THRESHOLD}) — reduce grade confidence')
    if has_glare:
        cv_issues.append(f'surface glare on {glare_fraction*100:.1f}% of card area — may hide scratches or haze')
    if brightness_mean < 50:
        cv_issues.append('very dark image — poor lighting may obscure surface defects')
    elif brightness_mean > 220 and not has_glare:
        cv_issues.append('overexposed image — highlight detail may be lost')

    return dict(blur_score=round(blur_score,1), glare_fraction=round(glare_fraction,4),
                brightness_mean=round(brightness_mean,1), brightness_std=round(brightness_std,1),
                is_blurry=is_blurry, has_glare=has_glare, cv_issues=cv_issues)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2a — corner whitening
# ══════════════════════════════════════════════════════════════════════════════

def _corner_patch(gray_canonical: np.ndarray, r0: int, c0: int, ph: int, pw: int) -> dict:
    patch          = gray_canonical[r0:r0+ph, c0:c0+pw]
    brightness     = float(patch.mean())
    white_fraction = float((patch > WHITE_THRESH).mean())
    return dict(brightness=round(brightness,1),
                white_fraction=round(white_fraction,4),
                whitening=white_fraction > WHITENING_THRESHOLD)


def analyse_corners(img_rgb: np.ndarray) -> dict:
    """
    Per-corner whitening analysis on the canonical (384×544) grayscale image.

    Matches cvDetectors.ts computeCornerAnalysis().
    Each corner is a CORNER_W_PX × CORNER_H_PX patch.
    Returns TL/TR/BL/BR CornerPatch dicts + any_whitening + whitening_corners.
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray      = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY)
    H, W      = gray.shape
    pw, ph    = CORNER_W_PX, CORNER_H_PX

    patches = {
        'TL': _corner_patch(gray, 0,      0,      ph, pw),
        'TR': _corner_patch(gray, 0,      W - pw, ph, pw),
        'BL': _corner_patch(gray, H - ph, 0,      ph, pw),
        'BR': _corner_patch(gray, H - ph, W - pw, ph, pw),
    }
    whitening_corners = [k for k, v in patches.items() if v['whitening']]
    return dict(**patches,
                any_whitening=len(whitening_corners) > 0,
                whitening_corners=whitening_corners)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2b — Detector A: Border Irregularity
# ══════════════════════════════════════════════════════════════════════════════

def estimate_glare_mask(gray: np.ndarray) -> np.ndarray:
    """Return boolean mask of glare pixels (>= SURFACE_GLARE_THRESH)."""
    return gray >= SURFACE_GLARE_THRESH


def _make_border_masks(H: int, W: int):
    """
    Return corner-exclusive side masks + 'corners' mask + 'all' mask.
    top/bottom: rows [1,BORDER_BAND_H) and [H-BORDER_BAND_H,H-1),
                cols [BORDER_BAND_W, W-BORDER_BAND_W)  (corner-exclusive)
    left/right: rows [BORDER_BAND_H, H-BORDER_BAND_H),
                cols [1,BORDER_BAND_W) and [W-BORDER_BAND_W,W-1)
    """
    top = np.zeros((H, W), dtype=bool)
    top[1:BORDER_BAND_H, BORDER_BAND_W:W-BORDER_BAND_W] = True

    bottom = np.zeros((H, W), dtype=bool)
    bottom[H-BORDER_BAND_H:H-1, BORDER_BAND_W:W-BORDER_BAND_W] = True

    left = np.zeros((H, W), dtype=bool)
    left[BORDER_BAND_H:H-BORDER_BAND_H, 1:BORDER_BAND_W] = True

    right = np.zeros((H, W), dtype=bool)
    right[BORDER_BAND_H:H-BORDER_BAND_H, W-BORDER_BAND_W:W-1] = True

    corners = np.zeros((H, W), dtype=bool)
    corners[:BORDER_BAND_H, :BORDER_BAND_W] = True
    corners[:BORDER_BAND_H, W-BORDER_BAND_W:] = True
    corners[H-BORDER_BAND_H:, :BORDER_BAND_W] = True
    corners[H-BORDER_BAND_H:, W-BORDER_BAND_W:] = True

    all_band = top | bottom | left | right | corners
    return dict(top=top, bottom=bottom, left=left, right=right, corners=corners, all=all_band)


def analyse_border_irregularity(img_rgb: np.ndarray) -> dict:
    """
    Detector A — Border Irregularity.

    Sobel gradient + brightness whitening + connected components in the
    card border band. Side stats are corner-exclusive.
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray      = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W      = gray.shape

    sobel_x   = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y   = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

    masks = _make_border_masks(H, W)

    def side_stat(mask):
        if not mask.any():
            return dict(grad_fraction=0.0, bright_fraction=0.0, mean_mag=0.0)
        mag_vals   = magnitude[mask]
        gray_vals  = gray[mask]
        count      = mask.sum()
        grad_frac  = float((mag_vals > BORDER_GRAD_THRESHOLD).sum()) / count
        bright_frac = float((gray_vals > BORDER_WHITE_THRESH).sum()) / count
        mean_mag   = float(mag_vals.mean())
        return dict(grad_fraction=round(grad_frac, 4),
                    bright_fraction=round(bright_frac, 4),
                    mean_mag=round(mean_mag, 1))

    per_side = {
        'top':    side_stat(masks['top']),
        'bottom': side_stat(masks['bottom']),
        'left':   side_stat(masks['left']),
        'right':  side_stat(masks['right']),
    }

    all_mask     = masks['all']
    all_mag      = magnitude[all_mask]
    all_gray     = gray[all_mask]
    total_count  = all_mask.sum()
    anomaly_mask_2d = (magnitude > BORDER_GRAD_THRESHOLD) & all_mask

    total_grad_fraction = round(float(anomaly_mask_2d.sum()) / max(total_count, 1), 4)
    bright_fraction     = round(float((all_gray > BORDER_WHITE_THRESH).sum()) / max(total_count, 1), 4)

    # Connected components on all-band anomaly map
    binary_u8 = anomaly_mask_2d.astype(np.uint8)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary_u8, connectivity=8)
    # stats[0] = background, skip it
    areas = [int(stats[i, cv2.CC_STAT_AREA]) for i in range(1, num_labels) if stats[i, cv2.CC_STAT_AREA] > 1]
    component_count     = len(areas)
    max_component_area  = max(areas) if areas else 0
    mean_component_area = round(sum(areas) / len(areas), 1) if areas else 0.0

    score = (
        0.40 * min(total_grad_fraction / 0.30, 1.0)
        + 0.20 * min(bright_fraction   / 0.20, 1.0)
        + 0.25 * min(component_count   / 40.0, 1.0)
        + 0.15 * min(max_component_area / 80.0, 1.0)
    )
    score = round(score, 4)

    severity = ('heavy'    if score >= 0.55 else
                'moderate' if score >= 0.30 else
                'light'    if score >= 0.12 else 'none')

    return dict(
        score=score, severity=severity,
        total_grad_fraction=total_grad_fraction,
        bright_fraction=bright_fraction,
        component_count=component_count,
        max_component_area=max_component_area,
        mean_component_area=mean_component_area,
        per_side=per_side,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2c — Detector B: Surface Lines
# ══════════════════════════════════════════════════════════════════════════════

def make_interior_mask(H: int, W: int) -> np.ndarray:
    """Boolean mask for the card interior (excluding border margins)."""
    mask = np.zeros((H, W), dtype=bool)
    mask[SURFACE_MARGIN_H:H-SURFACE_MARGIN_H, SURFACE_MARGIN_W:W-SURFACE_MARGIN_W] = True
    return mask


def analyse_surface_line_detector(img_rgb: np.ndarray) -> dict:
    """
    Detector B — Surface Lines.

    Top-hat morphology (H/V/diagonal kernels) + Hough lines in card interior.
    Detects scratches, holo damage, and print lines via directional energy.
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray      = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W      = gray.shape

    interior_mask = make_interior_mask(H, W)

    # Glare fraction in interior
    glare_mask    = gray >= SURFACE_GLARE_THRESH
    interior_total = interior_mask.sum()
    glare_count   = int((glare_mask & interior_mask).sum())
    glare_fraction = round(glare_count / max(interior_total, 1), 4)

    # Sobel in interior (non-glare pixels)
    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

    valid_mask   = interior_mask & ~glare_mask
    valid_count  = int(valid_mask.sum())

    sum_abs_gx = float(np.abs(sobel_x[valid_mask]).sum())
    sum_abs_gy = float(np.abs(sobel_y[valid_mask]).sum())
    mean_abs_gx = sum_abs_gx / max(valid_count, 1)
    mean_abs_gy = sum_abs_gy / max(valid_count, 1)
    energy_imbalance = abs(mean_abs_gx - mean_abs_gy) / (mean_abs_gx + mean_abs_gy + 1e-6)

    high_mag_mask = valid_mask & (magnitude > SURFACE_GRAD_THRESHOLD)
    total_high_mag = int(high_mag_mask.sum())

    if total_high_mag > 0:
        angles_rad = np.abs(np.arctan2(sobel_y[high_mag_mask], sobel_x[high_mag_mask]))
        angles_deg = np.degrees(angles_rad)
        horiz_count = int(((angles_deg < 20) | (angles_deg > 160)).sum())
        vert_count  = int(((angles_deg > 70) & (angles_deg < 110)).sum())
        diag_count  = total_high_mag - horiz_count - vert_count
    else:
        horiz_count = vert_count = diag_count = 0

    diagonal_energy_fraction = round(diag_count  / max(total_high_mag, 1), 4)
    h_energy_fraction        = round(horiz_count / max(total_high_mag, 1), 4)
    v_energy_fraction        = round(vert_count  / max(total_high_mag, 1), 4)
    energy_imbalance         = round(energy_imbalance, 4)

    # Hough line detection for additional diagonal scoring
    gray_u8   = gray.astype(np.uint8)
    edges     = cv2.Canny(gray_u8, 50, 150)
    interior_u8 = interior_mask.astype(np.uint8)
    edges_interior = cv2.bitwise_and(edges, edges, mask=interior_u8)
    lines = cv2.HoughLinesP(edges_interior, 1, np.pi/180, threshold=30,
                             minLineLength=40, maxLineGap=10)
    hough_diag_lines = 0
    hough_total_lines = 0
    if lines is not None:
        hough_total_lines = len(lines)
        for line in lines:
            x1, y1, x2, y2 = line[0]
            dx, dy = x2 - x1, y2 - y1
            if dx == 0 and dy == 0:
                continue
            angle = abs(np.degrees(np.arctan2(dy, dx)))
            if 20 <= angle <= 70 or 110 <= angle <= 160:
                hough_diag_lines += 1

    score = (
        0.50 * min(diagonal_energy_fraction * 2.5, 1.0)
        + 0.30 * min(energy_imbalance * 2.0, 1.0)
        + 0.20 * min(total_high_mag / (valid_count * 0.25 + 1), 1.0)
    )
    score = round(score, 4)

    severity = ('heavy'    if score >= 0.50 else
                'moderate' if score >= 0.28 else
                'light'    if score >= 0.12 else 'none')

    confidence = ('low'    if glare_fraction > 0.15 else
                  'medium' if glare_fraction > 0.05 else 'high')

    return dict(
        score=score, severity=severity, confidence=confidence,
        glare_fraction=glare_fraction,
        diagonal_energy_fraction=diagonal_energy_fraction,
        h_energy_fraction=h_energy_fraction,
        v_energy_fraction=v_energy_fraction,
        energy_imbalance=energy_imbalance,
        hough_diag_lines=hough_diag_lines,
        hough_total_lines=hough_total_lines,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  Prompt formatters
# ══════════════════════════════════════════════════════════════════════════════

def format_cv_section(cv: dict | None) -> str:
    if cv is None:
        return ''
    issue_str  = '\n'.join(f'  • {s}' for s in cv['cv_issues']) if cv['cv_issues'] else '  • none detected'
    blur_note  = '\nNOTE: Image is blurry — lower confidence, flag in issues.'  if cv['is_blurry'] else ''
    glare_note = '\nNOTE: Significant glare — surface haze or scratches may be hidden.' if cv['has_glare'] else ''
    return (f'─── CV MEASUREMENTS (pixel analysis, run before this prompt) ───\n'
            f'Blur score   : {cv["blur_score"]}  (threshold ≥ {BLUR_THRESHOLD} = acceptably sharp)\n'
            f'Glare        : {cv["glare_fraction"]*100:.1f}% pixels overexposed  (≤ 5% acceptable)\n'
            f'Brightness   : mean {cv["brightness_mean"]} / std {cv["brightness_std"]}  (0–255 scale)\n'
            f'CV issues    :\n{issue_str}{blur_note}{glare_note}\n\n')


def format_corner_section(corners: dict | None) -> str:
    if corners is None:
        return ''
    def fmt(p):
        w = f"  ⚠ WHITENING" if p['whitening'] else ''
        return f"brightness {p['brightness']}, white {p['white_fraction']*100:.1f}%{w}"
    note = (f"NOTE: Whitening detected at {', '.join(corners['whitening_corners'])} — "
            f"these corners cannot grade PSA 10; likely caps at PSA 8 or below."
            if corners['any_whitening']
            else "NOTE: No corner whitening detected in pixel analysis.")
    return (f'─── CV CORNER ANALYSIS (pixel measurement) ───\n'
            f'  TL: {fmt(corners["TL"])}\n'
            f'  TR: {fmt(corners["TR"])}\n'
            f'  BL: {fmt(corners["BL"])}\n'
            f'  BR: {fmt(corners["BR"])}\n'
            f'{note}\n\n')


def format_border_irregularity_section(border_irr: dict | None) -> str:
    if border_irr is None:
        return ''
    s  = border_irr
    ps = s['per_side']
    note = ('Elevated border anomaly — inspect for whitening, chips, edge roughness, or print artifacts.'
            if s['severity'] != 'none'
            else 'Border band appears clean.')
    return (
        f'─── CV DETECTOR A: BORDER IRREGULARITY ───\n'
        f'  Score: {s["score"]}  Severity: {s["severity"].upper()}\n'
        f'  Gradient: {s["total_grad_fraction"]*100:.1f}%  '
        f'Bright: {s["bright_fraction"]*100:.1f}%  '
        f'Clusters: {s["component_count"]}  '
        f'Largest: {s["max_component_area"]}px\n'
        f'  Per-side grad%: '
        f'top={ps["top"]["grad_fraction"]*100:.1f}% '
        f'bottom={ps["bottom"]["grad_fraction"]*100:.1f}% '
        f'left={ps["left"]["grad_fraction"]*100:.1f}% '
        f'right={ps["right"]["grad_fraction"]*100:.1f}%\n'
        f'NOTE: {note}\n\n'
    )


def format_surface_lines_section(surface_ln: dict | None) -> str:
    if surface_ln is None:
        return ''
    s = surface_ln
    note = ('Directional surface features detected — possible scratches, holo damage, or print lines.'
            if s['severity'] != 'none'
            else 'No dominant directional surface features detected.')
    glare_note = '\nNOTE: High glare limits surface analysis reliability.' if s['confidence'] == 'low' else ''
    return (
        f'─── CV DETECTOR B: SURFACE LINES ───\n'
        f'  Score: {s["score"]}  Severity: {s["severity"].upper()}  Confidence: {s["confidence"]}\n'
        f'  Diagonal: {s["diagonal_energy_fraction"]*100:.0f}%  '
        f'H: {s["h_energy_fraction"]*100:.0f}%  '
        f'V: {s["v_energy_fraction"]*100:.0f}%  '
        f'Energy imbalance: {s["energy_imbalance"]}  '
        f'Glare: {s["glare_fraction"]*100:.1f}%\n'
        f'NOTE: {note}{glare_note}\n\n'
    )



# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2d — Detector C: FFT Surface Directionality
# ══════════════════════════════════════════════════════════════════════════════

def analyse_surface_fft_directionality(img_rgb: np.ndarray) -> dict:
    """
    Detector C — FFT Surface Directionality.

    Measures whether the card interior has unusual directional energy in the
    frequency domain. A clean surface should be roughly isotropic; scratches,
    print lines, or holo damage create directional energy spikes.

    FFT-space → spatial-domain mapping (important — counterintuitive):
      FFT vertical axis   (angle ≈ 90°)  = horizontal lines in image
      FFT horizontal axis (angle ≈ 0°)   = vertical lines in image
      FFT diagonal axes   (angle ≈ 45°)  = diagonal lines in image

    Directionality score: ~1.0 = isotropic, > 2.5 = strongly directional.
    Analysis restricted to mid-frequency ring (rr 10–80) to exclude coarse
    artwork structure (low freq) and compression noise (high freq).
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H),
                           interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W = gray.shape

    interior_mask = make_interior_mask(H, W)
    interior = gray.copy()
    interior[~interior_mask] = float(interior[interior_mask].mean()) if interior_mask.sum() else 0.0

    # Hanning window suppresses spectral leakage from the rectangular interior boundary
    window   = np.outer(np.hanning(H), np.hanning(W))
    interior = interior * window

    F   = np.fft.fftshift(np.fft.fft2(interior))
    mag = np.log1p(np.abs(F))

    cy, cx = H // 2, W // 2
    yy, xx = np.mgrid[0:H, 0:W]
    dx = xx - cx
    dy = yy - cy
    angle = np.degrees(np.arctan2(dy, dx))   # range [-180, 180]
    rr    = np.sqrt(dx**2 + dy**2)

    # Mid-frequency ring only: excludes DC + very-low-freq (coarse artwork) and
    # high-freq (compression noise). Real scratches live in rr 10–80.
    freq_mask = (rr > 10) & (rr < 80)

    # Directional bands in FFT space (±15° around each axis)
    # NOTE: labels below are FFT-space; spatial-domain is perpendicular
    fft_h_band = (np.abs(angle) < 15) | (np.abs(angle) > 165)         # → vertical lines in image
    fft_v_band = np.abs(np.abs(angle) - 90) < 15                       # → horizontal lines in image
    fft_d_band = ((np.abs(np.abs(angle) - 45)  < 15) |
                  (np.abs(np.abs(angle) - 135) < 15))                  # → diagonal lines in image

    h_energy    = float(mag[fft_h_band & freq_mask].sum())
    v_energy    = float(mag[fft_v_band & freq_mask].sum())
    diag_energy = float(mag[fft_d_band & freq_mask].sum())
    total_energy = float(mag[freq_mask].sum()) + 1e-6

    # Normalise to fractions so values are comparable across images
    h_frac    = h_energy    / total_energy
    v_frac    = v_energy    / total_energy
    diag_frac = diag_energy / total_energy

    # Directionality score: dominant band / mean band
    # ~1.0 = isotropic (normal card), > 2.5 = one direction strongly dominates
    dominant  = max(h_frac, v_frac, diag_frac)
    mean_band = (h_frac + v_frac + diag_frac) / 3
    dir_score = dominant / (mean_band + 1e-6)

    # Dominant axis in spatial domain (perpendicular to FFT dominant)
    if dominant == h_frac:
        dominant_spatial = 'vertical'      # FFT horizontal axis = vertical lines in image
    elif dominant == v_frac:
        dominant_spatial = 'horizontal'    # FFT vertical axis   = horizontal lines in image
    else:
        dominant_spatial = 'diagonal'

    return {
        'fft_directionality_score': round(float(dir_score),  3),
        'fft_dominant_axis':        dominant_spatial,
        'fft_h_fraction':           round(h_frac,    4),    # FFT horiz → vertical lines
        'fft_v_fraction':           round(v_frac,    4),    # FFT vert  → horizontal lines
        'fft_diag_fraction':        round(diag_frac, 4),    # FFT diag  → diagonal lines
        'fft_hv_ratio':             round(h_energy / (v_energy + 1e-6), 3),
    }


# ══════════════════════════════════════════════════════════════════════════════
#  Combined damage detector
# ══════════════════════════════════════════════════════════════════════════════

def analyse_card_damage(img_rgb: np.ndarray) -> dict:
    """
    Runs all three damage detectors on a single image and returns a combined dict.

      border_detector  — Detector A: border irregularity (Sobel + CC, per-side)
      surface_detector — Detector B: surface lines (top-hat + Hough, interior)
      surface_fft      — Detector C: FFT directionality (global energy, interior)

    The three detectors are complementary:
      A catches physical border damage visible at low-mid spatial frequency.
      B catches linear surface features (scratches) via morphology + Hough.
      C catches global directional energy — diffuse scratches B may miss.
    B + C agreement raises scratch confidence; A + C divergence suggests
    border-only damage with a clean surface.
    """
    return {
        'border_detector':  analyse_border_irregularity(img_rgb),
        'surface_detector': analyse_surface_line_detector(img_rgb),
        'surface_fft':      analyse_surface_fft_directionality(img_rgb),
    }


def format_fft_section(fft: dict | None) -> str:
    if not fft:
        return ''
    score = fft['fft_directionality_score']
    axis  = fft['fft_dominant_axis']
    # Severity heuristic: > 2.5 = suspicious, > 1.8 = borderline
    note = (
        f'Strongly directional surface energy detected ({axis} dominant) — '
        'consistent with scratches, print lines, or holo damage. '
        'Cross-check with Detector B surface_lines result.'
        if score > 2.5 else
        f'Moderate directional energy ({axis} dominant) — borderline; '
        'may reflect artwork structure rather than damage.'
        if score > 1.8 else
        'Surface energy is approximately isotropic — no dominant directional anomaly.'
    )
    return (
        f'\u2500\u2500\u2500 CV DETECTOR C: FFT SURFACE DIRECTIONALITY \u2500\u2500\u2500\n'
        f'  Directionality score: {score:.3f}  (\u22481.0 = isotropic, >2.5 = suspicious)\n'
        f'  Dominant axis (spatial): {axis}\n'
        f'  Mid-freq energy fractions \u2014 '
        f'H: {fft["fft_h_fraction"]*100:.1f}%  '
        f'V: {fft["fft_v_fraction"]*100:.1f}%  '
        f'Diag: {fft["fft_diag_fraction"]*100:.1f}%\n'
        f'NOTE: {note}\n\n'
    )


def format_centering_section(centering: dict | None) -> str:
    if centering is None:
        return ''
    method = 'perspective-warped image' if centering['warp_applied'] else 'uncorrected image (warp failed)'
    px     = centering['border_px']
    reliable_note = '' if centering['reliable'] else '\nNOTE: Warp detection failed — centering is approximate.'
    return (f'─── CV CENTERING MEASUREMENT ({method}) ───\n'
            f'  L/R: {centering["lr_ratio"]}  T/B: {centering["tb_ratio"]}\n'
            f'  Border px: top={px["top"]} bottom={px["bottom"]} '
            f'left={px["left"]} right={px["right"]}\n'
            f'NOTE: Use this as ground truth for centering. '
            f'Deviations from 50/50 directly affect PSA grade ceiling.{reliable_note}\n\n')


def format_side_labels(sides: list[str]) -> str:
    if all(s == 'unknown' for s in sides):
        return ''
    lines = '\n'.join(f'  Image {i+1}: {s.upper()}' for i, s in enumerate(sides))
    return (f'─── IMAGE SIDE CLASSIFICATION (pre-classified by pixel analysis) ───\n'
            f'{lines}\n'
            f'NOTE: Trust these labels — do NOT reclassify images yourself.\n\n')


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — side classifier
# ══════════════════════════════════════════════════════════════════════════════

def classify_card_side(img_rgb: np.ndarray) -> str:
    """HSV blue-band classifier for Pokémon card back. Port of classifyCardSide()."""
    try:
        small = cv2.resize(img_rgb, (128, 128), interpolation=cv2.INTER_AREA)
        hsv   = cv2.cvtColor(small, cv2.COLOR_RGB2HSV).astype(np.float32)
        h     = hsv[:, :, 0] * 2.0
        s     = hsv[:, :, 1] / 255.0
        v     = hsv[:, :, 2] / 255.0
        mask  = (h >= BACK_HUE_MIN) & (h <= BACK_HUE_MAX) & (s >= BACK_SAT_MIN) & (v >= BACK_VAL_MIN)
        return 'back' if mask.sum() / (128 * 128) >= BACK_PIXEL_THRESHOLD else 'front'
    except Exception as exc:
        print(f'  classify_card_side failed: {exc}')
        return 'unknown'


# ══════════════════════════════════════════════════════════════════════════════
#  Helper loaders (used by grade_with_claude and visualisers)
# ══════════════════════════════════════════════════════════════════════════════

def _fetch_buffer(url: str) -> np.ndarray | None:
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        return np.array(Image.open(io.BytesIO(resp.content)).convert('RGB'))
    except Exception as exc:
        print(f'  fetch failed: {url[:60]}... ({exc})')
        return None


def _load_buffer(path: str) -> np.ndarray | None:
    try:
        return np.array(Image.open(path).convert('RGB'))
    except Exception as exc:
        print(f'  load failed: {path} ({exc})')
        return None


def run_cv_detectors(image_urls: list[str]) -> dict | None:
    """Phase 1 only (quality). Tries first 2 URLs."""
    for url in image_urls[:2]:
        try:
            img = _fetch_buffer(url)
            if img is not None:
                result = _analyse_image_array(img)
                result['_source_url'] = url
                return result
        except Exception as exc:
            print(f'  CV: skipping {url[:70]}... ({exc})')
    return None


def run_cv_detectors_local(image_path: str) -> dict | None:
    try:
        img = _load_buffer(image_path)
        if img is None:
            return None
        result = _analyse_image_array(img)
        result['_source_url'] = image_path
        return result
    except Exception as exc:
        print(f'  CV local: failed ({exc})')
        return None


print('✅ CV detectors: A (border_irregularity) + B (surface_lines) + C (fft_directionality) + analyse_card_damage()')

### Side Classifier Test

Visualise the HSV blue-band mask on your local images to verify the classifier is working correctly before running a full grading call.

In [ ]:
def visualise_side_classifier(image_paths: list[str] | None = None,
                               image_urls:  list[str] | None = None) -> None:
    """
    Show each image alongside its HSV blue-band mask and classification result.
    Useful for tuning thresholds or debugging misclassifications.
    """
    sources = image_paths or image_urls or []
    imgs: list[np.ndarray] = []
    for src in sources:
        arr = _load_buffer(src) if image_paths else _fetch_buffer(src)
        if arr is not None:
            imgs.append(arr)

    if not imgs:
        print('No images loaded')
        return

    n = len(imgs)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 7))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col, img in enumerate(imgs):
        small = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
        hsv   = cv2.cvtColor(small, cv2.COLOR_RGB2HSV).astype(np.float32)
        h = hsv[:, :, 0] * 2.0
        s = hsv[:, :, 1] / 255.0
        v = hsv[:, :, 2] / 255.0

        mask = (
            (h >= BACK_HUE_MIN) & (h <= BACK_HUE_MAX) &
            (s >= BACK_SAT_MIN) &
            (v >= BACK_VAL_MIN)
        )
        blue_frac = mask.sum() / (128 * 128)
        label     = classify_card_side(img)
        colour    = '#10b981' if label == 'back' else '#f97316'

        # Top row: original
        axes[0, col].imshow(img)
        axes[0, col].set_title(
            f'{label.upper()}  ({blue_frac*100:.1f}% blue)',
            color=colour, fontsize=11, fontweight='bold')
        axes[0, col].axis('off')

        # Bottom row: mask overlay
        overlay = small.copy()
        overlay[mask] = [0, 200, 100]
        axes[1, col].imshow(overlay)
        axes[1, col].set_title('Blue-band mask', color='white', fontsize=9)
        axes[1, col].axis('off')

    plt.suptitle(
        f'Threshold: >{BACK_PIXEL_THRESHOLD*100:.0f}% blue pixels → BACK',
        color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# ── Run the test on your local images ─────────────────────────────────────────
# Change to image_urls=['url1','url2'] to test with eBay URLs instead.
TEST_CLASSIFY_PATHS = ['image0_front.jpeg', 'image0_back.jpeg']

if all(Path(p).exists() for p in TEST_CLASSIFY_PATHS):
    visualise_side_classifier(image_paths=TEST_CLASSIFY_PATHS)
else:
    print('Local test images not found — set TEST_CLASSIFY_PATHS or use image_urls')

### Phase 2 CV Visualiser — Corners, Centering, Scratch

Run this on any local image to see the full structural analysis before sending to Claude.

In [ ]:
def visualise_cv_pipeline(image_path: str | None = None,
                           image_url:  str | None = None) -> None:
    """
    Full Phase 2 CV visualisation for a single card image:
    - Corner patches with whitening overlay
    - Border-band Sobel gradient mask
    - Perspective-warped card with centering border lines
    """
    img = _load_buffer(image_path) if image_path else _fetch_buffer(image_url)
    if img is None:
        print('Failed to load image')
        return

    # Run all Phase 2 detectors
    corners   = analyse_corners(img)
    scratch   = analyse_border_anomaly_score(img)
    centering = measure_centering(img)
    side      = classify_card_side(img)

    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0d1117')

    # ── Row 1: Original | Warped | Sobel border mask ──────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    ax1.imshow(img)
    ax1.set_title(f'Original  [{side.upper()}]', color='white')
    ax1.axis('off')

    warped, warp_ok = perspective_warp(img)
    ax2 = fig.add_subplot(2, 3, 2)
    if warp_ok and warped is not None:
        ax2.imshow(warped)
        # Draw centering lines
        c  = centering
        px = c['border_px']
        H, W = warped.shape[:2]
        ax2.axhline(px['top'],        color='#10b981', lw=1.5, alpha=0.8, label=f"top={px['top']}")
        ax2.axhline(H - px['bottom'], color='#10b981', lw=1.5, alpha=0.8, label=f"bot={px['bottom']}")
        ax2.axvline(px['left'],       color='#6366f1', lw=1.5, alpha=0.8, label=f"left={px['left']}")
        ax2.axvline(W - px['right'],  color='#6366f1', lw=1.5, alpha=0.8, label=f"right={px['right']}")
        ax2.set_title(f"Warped  L/R {c['lr_ratio']}  T/B {c['tb_ratio']}", color='#10b981')
    else:
        ax2.text(0.5, 0.5, 'Warp failed\n(no clear card quad found)',
                 ha='center', va='center', color='#f97316', fontsize=11,
                 transform=ax2.transAxes)
        ax2.set_title('Centering (warp failed)', color='#f97316')
    ax2.axis('off')

    # Sobel border-band mask
    canonical = cv2.resize(img, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray_c    = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    sx        = cv2.Sobel(gray_c, cv2.CV_32F, 1, 0, ksize=3)
    sy        = cv2.Sobel(gray_c, cv2.CV_32F, 0, 1, ksize=3)
    mag       = np.sqrt(sx**2 + sy**2)
    H_c, W_c  = canonical.shape[:2]
    band_mask = np.zeros((H_c, W_c), dtype=bool)
    band_mask[:BORDER_BAND_H, :]  = True
    band_mask[-BORDER_BAND_H:, :] = True
    band_mask[:, :BORDER_BAND_W]  = True
    band_mask[:, -BORDER_BAND_W:] = True

    viz_sobel = canonical.copy()
    edge_mask = (mag > BORDER_ANOMALY_GRADIENT_THRESHOLD) & band_mask
    viz_sobel[edge_mask]          = [255, 80, 80]   # red = high gradient in border
    viz_sobel[band_mask & ~edge_mask] = (viz_sobel[band_mask & ~edge_mask] * 0.5 + np.array([0, 80, 150]) * 0.5).astype(np.uint8)

    ax3 = fig.add_subplot(2, 3, 3)
    ax3.imshow(viz_sobel)
    ax3.set_title(f"Scratch indicator  {scratch['severity'].upper()}  ({scratch['edge_pixel_count']} px)",
                  color='#ef4444' if scratch['severity'] != 'none' else '#10b981')
    ax3.axis('off')

    # ── Row 2: Four corner patches ────────────────────────────────────────────
    corner_positions = [('TL', 2, 3, 4), ('TR', 2, 3, 5), ('BL', 2, 3, 6)]  # 3 fit; BR in title
    gray_can_uint8 = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY)
    corner_imgs = {
        'TL': canonical[:CORNER_H_PX, :CORNER_W_PX],
        'TR': canonical[:CORNER_H_PX, -CORNER_W_PX:],
        'BL': canonical[-CORNER_H_PX:, :CORNER_W_PX],
        'BR': canonical[-CORNER_H_PX:, -CORNER_W_PX:],
    }
    for col_idx, (name, _, _, subplot_idx) in enumerate(corner_positions):
        ax = fig.add_subplot(2, 3, subplot_idx)
        patch_img = corner_imgs[name].copy()
        p = corners[name]
        # Highlight white pixels
        white_mask = cv2.cvtColor(patch_img, cv2.COLOR_RGB2GRAY) > WHITE_THRESH
        patch_img[white_mask] = [255, 100, 100] if p['whitening'] else [100, 200, 100]
        ax.imshow(patch_img)
        color = '#ef4444' if p['whitening'] else '#10b981'
        ax.set_title(f"{name}  {p['brightness']:.0f}bri  {p['white_fraction']*100:.1f}%white"
                     + (' ⚠' if p['whitening'] else ''), color=color, fontsize=9)
        ax.axis('off')

    # BR in a text box
    ax_br = fig.add_subplot(2, 3, 6) if len(corner_positions) < 4 else None
    if ax_br:
        br = corners['BR']
        patch_img = corner_imgs['BR'].copy()
        white_mask = cv2.cvtColor(patch_img, cv2.COLOR_RGB2GRAY) > WHITE_THRESH
        patch_img[white_mask] = [255, 100, 100] if br['whitening'] else [100, 200, 100]
        ax_br.imshow(patch_img)
        color = '#ef4444' if br['whitening'] else '#10b981'
        ax_br.set_title(f"BR  {br['brightness']:.0f}bri  {br['white_fraction']*100:.1f}%white"
                        + (' ⚠' if br['whitening'] else ''), color=color, fontsize=9)
        ax_br.axis('off')

    summary = (f"Corners: {'⚠ ' + ', '.join(corners['whitening_corners']) if corners['any_whitening'] else '✓ clean'}  |  "
               f"Centering: L/R {centering['lr_ratio']}  T/B {centering['tb_ratio']}  |  "
               f"Surface: {scratch['severity']}")
    fig.suptitle(summary, color='white', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    # Print injected CV sections
    print('\n' + '─'*65)
    print('PROMPT SECTIONS THAT WOULD BE INJECTED:')
    print('─'*65)
    print(format_corner_section(corners), end='')
    print(format_border_anomaly_section(scratch), end='')
    print(format_centering_section(centering), end='')


# ── Run on your local images ──────────────────────────────────────────────────
VIZ_PATH = 'image0_front.jpeg'   # change to your image

if Path(VIZ_PATH).exists():
    visualise_cv_pipeline(image_path=VIZ_PATH)
else:
    print(f'Set VIZ_PATH to an existing image file (tried: {VIZ_PATH})')

## Image Helpers

Port of `fetchResized()` from `claudeVision.ts` — downloads and resizes the primary image to ≤ 1024 px for fewer Claude input tokens.

In [ ]:
def fetch_resized(url: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Port of fetchResized() from claudeVision.ts.

    Downloads url, resizes longest edge to ≤ max_size (no upscaling),
    encodes as JPEG base64.  Returns None on any failure.
    """
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  resize failed: {exc}')
        return None


def load_local_image(path: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Same as fetch_resized but from a local file path."""
    try:
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  local load failed: {exc}')
        return None


def show_images(image_urls: list[str] | None = None,
                local_paths: list[str] | None = None,
                max_show: int = 6):
    """Display thumbnail grid of the test images."""
    sources = []
    if image_urls:
        for url in image_urls[:max_show]:
            try:
                resp = requests.get(url.replace('s-l1600', 's-l300'), timeout=6)
                sources.append(Image.open(io.BytesIO(resp.content)).convert('RGB'))
            except Exception:
                pass
    if local_paths:
        for p in local_paths[:max_show]:
            try:
                sources.append(Image.open(p).convert('RGB'))
            except Exception:
                pass

    if not sources:
        print('No images to display')
        return

    n = len(sources)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, img in zip(axes, sources):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'{n} image(s)', color='white', fontsize=11)
    plt.tight_layout()
    plt.show()


print('\u2705 Image helpers ready')

## Prompts

Copied verbatim from `src/lib/grading/claudeVision.ts`. **Edit these here to test prompt changes before deploying.**

In [ ]:
SYSTEM_PROMPT = """You are an expert PSA pre-grading assistant specializing in Pokémon trading cards.

Your task is to estimate the plausible PSA grade range supported by the visible evidence. You are not guaranteeing a final PSA grade.

Core principles:
- Base your assessment only on what is actually visible in the images.
- Never assume unseen areas are clean.
- If glare, blur, angle, cropping, sleeve reflections, compression, or low resolution hide details, reduce confidence and mention the limitation.
- When evidence is incomplete, prefer a wider grade range and lower confidence.
- Distinguish between: (1) visible defects and (2) unknowns caused by image limitations.

Evaluate in this order:
1. IMAGE CLASSIFICATION — label each image as "front", "back", or "other" based on visible content. The front shows the card artwork; the back shows the Pokémon logo / back design.
2. FRONT ANALYSIS — if a front image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR — whitening, fraying, rounding
   - Edges: top, bottom, left, right — nicks, chips, whitening
   - Surface: scratches, print lines, holo damage, staining, gloss loss
3. BACK ANALYSIS — if a back image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR
   - Edges: top, bottom, left, right
   - Surface: print quality, scratches, staining
4. COMBINED GRADE — derive the final grade_estimate and combined issues from the worst-case evidence across both sides.
5. CARD IDENTITY — identify name/set/year/number from the front image only.

PSA grade guidance:
- PSA 10 Gem Mint: near-perfect, centering ~55/45 or better on front, four sharp corners, no visible defects, full gloss
- PSA 9  Mint: slight visible wear allowed, minor issues possible, strong overall presentation
- PSA 8  NM-MT: light corner/edge wear, possible very light scratches or print issues
- PSA 7  NM: visible but not severe corner/edge wear, possible light scratches or minor print defects
- PSA 6  EX-MT and below: increasingly obvious wear, scratches, edge damage, staining, creasing, or major defects

Front-only rule (CRITICAL):
- If no back image is identifiable, set analysis_mode to "front_only"
- Set back_analysis.assessable to false and leave all back_analysis issue arrays empty
- Set grade_estimate.confidence to "low" regardless of front image quality
- Set grade_estimate.limiting_factor to "front_only"
- Add exactly this string to grading_decision.caveats: "Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."
- Do NOT speculate about the back condition

Other special instructions:
- Do not double-count the same defect across front and back.
- Vintage Pokémon cards (1999–2003) should be judged with awareness of era, but never use era to override visible evidence.
- Holo surface must be treated conservatively when glare or angle prevents reliable inspection.
- Do not overclaim PSA 9 or PSA 10 when image quality is limited.

Respond ONLY with one valid JSON object. No markdown, no prose outside JSON.

Every "issues" object must use exactly these keys: centering, corners, edges, surface, other.
Each maps to an array of strings. Use [] when nothing is visible for that category.

Allowed values:
- "analysis_mode": "front_only" or "front_back"
- "image_quality.status": "good", "partial", or "poor"
- "card_identity.confidence": "high", "medium", or "low"
- "grade_estimate.confidence": "high", "medium", or "low"
- "grade_estimate.limiting_factor": "front_only", "image_quality", "visible_damage", or null
- "grading_decision.gradable_candidate": "yes", "maybe", or "no"
- "front_analysis.assessable" / "back_analysis.assessable": true or false

If a card_identity field is uncertain, use null rather than guessing."""


USER_PROMPT = """Analyze this Pokémon card from the provided image set and return JSON with EXACTLY this structure.

Example when both front and back are present:
{
  "analysis_mode": "front_back",
  "card_identity": {
    "name": "Charizard", "set": "Base Set", "year": "1999", "number": "4", "confidence": "high"
  },
  "image_quality": {
    "status": "good", "warnings": [],
    "front_present": true, "back_present": true,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": true
  },
  "front_analysis": {
    "assessable": true,
    "centering": "58/42 L/R, 54/46 T/B",
    "issues": {
      "centering": ["Slightly left-heavy, approximately 58/42 L/R"],
      "corners":   ["Light whitening on top-right corner"],
      "edges":     [],
      "surface":   ["Faint holo scratches visible at angle"],
      "other":     []
    }
  },
  "back_analysis": {
    "assessable": true,
    "centering": "50/50 L/R, 51/49 T/B",
    "issues": {
      "centering": [],
      "corners":   [],
      "edges":     ["Minor whitening on bottom edge"],
      "surface":   [],
      "other":     []
    }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-8", "confidence": "high", "limiting_factor": "visible_damage",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.04,
      "6": 0.08, "7": 0.30, "8": 0.40, "9": 0.12, "10": 0.03
    }
  },
  "issues": {
    "centering": ["Slightly left-heavy front (58/42 L/R)"],
    "corners":   ["Light whitening on top-right corner (front)"],
    "edges":     ["Minor whitening on bottom edge (back)"],
    "surface":   ["Faint holo scratches visible at angle (front)"],
    "other":     []
  },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Card shows light wear consistent with PSA 7-8. Holo scratches are the primary concern.",
    "caveats": []
  }
}

Example when only the front is present (front_only):
{
  "analysis_mode": "front_only",
  "card_identity": { "name": null, "set": null, "year": null, "number": null, "confidence": "low" },
  "image_quality": {
    "status": "partial", "warnings": [], "front_present": true, "back_present": false,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": false
  },
  "front_analysis": {
    "assessable": true,
    "centering": "55/45 L/R, 53/47 T/B",
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "back_analysis": {
    "assessable": false,
    "centering": null,
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-9", "confidence": "low", "limiting_factor": "front_only",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.05,
      "6": 0.10, "7": 0.22, "8": 0.30, "9": 0.22, "10": 0.08
    }
  },
  "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Front appears reasonably clean but back is not available for assessment.",
    "caveats": ["Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."]
  }
}

Rules:
- Output exactly one JSON object. Use exactly the top-level keys shown above.
- front_analysis and back_analysis each have their own "issues" object for per-side defects.
- "issues" at the top level is the COMBINED worst-case from both sides. Tag each item with "(front)" or "(back)".
- Do not guess hidden defects. Only report what is actually visible.
- If the back is missing: set analysis_mode "front_only", back_analysis.assessable false, grade confidence "low", limiting_factor "front_only", and add the caveat string exactly as shown.
- If glare/blur/angle limits visibility, mention in image_quality.warnings and the relevant side analysis.
- Use a wider grade range and lower confidence when evidence is incomplete.
- Be conservative about PSA 9 and PSA 10 if surface or back visibility is limited.
- Distribution values must sum approximately to 1.00.
- confidence "high": top-2 PSA grades >60% of mass. "medium": 40–60%. "low": image too limited."""


print('✅ Prompts loaded')
print(f'  System prompt: {len(SYSTEM_PROMPT):,} chars')
print(f'  User prompt:   {len(USER_PROMPT):,} chars')

## Main Grading Function

Exact port of `gradeWithClaude()` from `claudeVision.ts`.

In [ ]:
def _array_to_base64_jpeg(img_rgb: np.ndarray, max_size: int = 1024,
                           quality: int = 85) -> dict | None:
    """Resize (longest edge ≤ max_size) and encode as JPEG base64."""
    try:
        pil = Image.fromarray(img_rgb)
        w, h = pil.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            pil = pil.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        pil.save(buf, format='JPEG', quality=quality)
        return {'base64': base64.b64encode(buf.getvalue()).decode(),
                'media_type': 'image/jpeg', 'size': pil.size}
    except Exception as exc:
        print(f'  resize failed: {exc}')
        return None


def grade_with_claude(
    image_urls:      list[str] | None = None,
    local_paths:     list[str] | None = None,
    title:           str  = '',
    api_key:         str  = API_KEY,
    run_centering:   bool = True,   # perspective warp + border scan (front image only)
    run_corners:     bool = True,   # per-corner whitening
    run_damage_detectors: bool = True,   # Detector A + B + C: border, surface lines, FFT
    show_prompt:     bool = False,
    show_raw:        bool = False,
) -> dict:
    """
    Full grading pipeline — exact port of claudeVision.ts plus Python-only
    centering measurement (perspective warp).

    Pipeline:
      1. Load all images as numpy arrays (shared across all CV steps)
      2. Classify each as front / back / unknown
      3. Sort: fronts first, backs second
      4. Phase 1 CV (quality) on first front image
      5. Phase 2 CV (structure) on first front image:
           a. Corner whitening
           b. Border irregularity — Detector A (border-band Sobel + CC)
           c. Surface lines — Detector B (directional energy + Hough)
           d. Centering via perspective warp  [Python-only, not in TS backend]
      6. Build image blocks + inject all CV sections into prompt
      7. Call Claude Haiku (max_tokens=2048)
      8. Sanity guard
    """
    client  = anthropic.Anthropic(api_key=api_key)
    sources = image_urls or local_paths or []

    # ── Step 1: Load all images ────────────────────────────────────────────────
    print('  [1/6] Loading images...')
    arrays: list[np.ndarray | None] = [
        (_fetch_buffer(s) if image_urls else _load_buffer(s))
        for s in sources[:6]
    ]

    # ── Step 2: Classify ──────────────────────────────────────────────────────
    print('  [2/6] Classifying front/back...')
    sides = [classify_card_side(a) if a is not None else 'unknown' for a in arrays]
    for src, side in zip(sources[:6], sides):
        print(f'         {Path(src).name if local_paths else src[:50]} → {side}')

    # ── Step 3: Sort ──────────────────────────────────────────────────────────
    rank  = {'front': 0, 'back': 1, 'unknown': 2}
    order = sorted(range(len(arrays)), key=lambda i: rank[sides[i]])
    sorted_arrays = [arrays[i]  for i in order]
    sorted_sides  = [sides[i]   for i in order]
    sorted_srcs   = [sources[i] for i in order]

    # ── Step 4: Phase 1 — image quality ───────────────────────────────────────
    print('  [3/6] Running Phase 1 CV (quality)...')
    front_idx = next((i for i, s in enumerate(sorted_sides) if s == 'front'), 0)
    front_arr = sorted_arrays[front_idx]
    cv_quality = _analyse_image_array(front_arr) if front_arr is not None else None

    # ── Step 5: Phase 2 — structural analysis on front image ──────────────────
    print('  [4/6] Running Phase 2 CV (structure)...')
    corners   = analyse_corners(front_arr)           if (run_corners   and front_arr is not None) else None
    damage = analyse_card_damage(front_arr) if (run_damage_detectors and front_arr is not None) else None
    border_irr = damage['border_detector']  if damage else None
    surface_ln = damage['surface_detector'] if damage else None
    fft_dir    = damage['surface_fft']      if damage else None
    centering = measure_centering(front_arr)         if (run_centering and front_arr is not None) else None

    if centering:
        warp_status = 'warp ✓' if centering['warp_applied'] else 'warp ✗ (fallback)'
        print(f'         centering {centering["lr_ratio"]} L/R, {centering["tb_ratio"]} T/B  [{warp_status}]')
    if corners and corners['any_whitening']:
        print(f'         ⚠ whitening at: {corners["whitening_corners"]}')
    if border_irr:
        print(f'         [A] border: {border_irr["severity"]} (score {border_irr["score"]}, {border_irr["component_count"]} clusters)')
    if surface_ln:
        print(f'         [B] surface: {surface_ln["severity"]} (score {surface_ln["score"]}, conf {surface_ln["confidence"]})')
    if fft_dir:
        print(f'         [C] fft: dir_score {fft_dir["fft_directionality_score"]:.2f}, dominant={fft_dir["fft_dominant_axis"]}')

    # ── Step 6: Build image content blocks ────────────────────────────────────
    print('  [5/6] Building image blocks + prompt...')
    image_blocks = []

    if sorted_arrays[0] is not None:
        enc = _array_to_base64_jpeg(sorted_arrays[0])
        if enc:
            image_blocks.append({'type': 'image', 'source': {
                'type': 'base64', 'media_type': enc['media_type'], 'data': enc['base64']}})

    for i, src in enumerate(sorted_srcs[1:], start=1):
        arr = sorted_arrays[i]
        if arr is not None:
            enc = _array_to_base64_jpeg(arr)
            if enc:
                image_blocks.append({'type': 'image', 'source': {
                    'type': 'base64', 'media_type': enc['media_type'], 'data': enc['base64']}})
        elif image_urls:
            image_blocks.append({'type': 'image', 'source': {'type': 'url', 'url': src}})

    # Build prompt with all CV sections
    full_text = (
        format_cv_section(cv_quality)
        + format_corner_section(corners)
        + format_border_irregularity_section(border_irr)
        + format_surface_lines_section(surface_ln)
        + format_fft_section(fft_dir)
        + format_centering_section(centering)
        + format_side_labels(sorted_sides)
        + (f'Listing title (use as identity hint, but trust the image over the title): "{title}"\n\n' if title else '')
        + USER_PROMPT
    )

    if show_prompt:
        print('\n' + '='*70 + '\nSYSTEM PROMPT:\n' + '='*70)
        print(SYSTEM_PROMPT)
        print('\n' + '='*70 + '\nUSER TEXT BLOCK:\n' + '='*70)
        print(full_text)
        print('='*70 + '\n')

    # ── Step 7: Call Claude Haiku ─────────────────────────────────────────────
    print(f'  [6/6] Calling Claude Haiku ({len(image_blocks)} image block(s))...')
    response = client.messages.create(
        model='claude-haiku-4-5', max_tokens=2048, system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': [*image_blocks, {'type': 'text', 'text': full_text}]}]
    )

    raw = ''.join(b.text for b in response.content if b.type == 'text').strip()

    if show_raw:
        print('\n--- RAW RESPONSE ---\n' + raw + '\n--- END RAW ---\n')

    json_str = re.sub(r'^```(?:json)?\s*\n?', '', raw, flags=re.IGNORECASE)
    json_str = re.sub(r'\n?```\s*$', '', json_str).strip()

    try:
        result = json.loads(json_str)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'Claude returned non-JSON (output_tokens={response.usage.output_tokens}):\n{raw[:600]}'
        ) from exc

    # Normalise distribution
    dist  = result.get('grade_estimate', {}).get('distribution', {})
    total_prob = sum(dist.values())
    if total_prob > 0 and abs(total_prob - 1) > 0.02:
        for k in dist:
            dist[k] = round(dist[k] / total_prob, 4)

    # Sanity guard
    front_ok = result.get('front_analysis', {}).get('assessable', True)
    back_ok  = result.get('back_analysis',  {}).get('assessable', True)
    if not front_ok and not back_ok:
        result['grade_estimate']['confidence']           = 'low'
        result['grade_estimate']['limiting_factor']      = 'image_quality'
        result['grading_decision']['gradable_candidate'] = 'no'
        result.setdefault('grading_decision', {}).setdefault('caveats', []).append(
            'Neither front nor back image was assessable — unable to grade reliably.')
        print('  ⚠ Sanity guard: both sides non-assessable → confidence forced to low')

    # Attach metadata
    result['_cv']        = cv_quality
    result['_corners']   = corners
    result['_border_irregularity'] = border_irr
    result['_surface_lines'] = surface_ln
    result['_fft_directionality'] = fft_dir
    result['_centering'] = centering
    result['_sides']     = sorted_sides
    result['_usage']     = {'input_tokens':  response.usage.input_tokens,
                             'output_tokens': response.usage.output_tokens}
    return result


print('✅ grade_with_claude() ready (Detectors A + B + C via analyse_card_damage)')

## Display Helpers

In [ ]:
GRADE_COLORS = {
    10: '#10b981', 9: '#34d399', 8: '#6366f1',
    7:  '#818cf8', 6: '#f97316', 5: '#fb923c',
    4:  '#ef4444', 3: '#dc2626', 2: '#b91c1c', 1: '#7f1d1d',
}

ISSUE_CATEGORY_LABELS = {
    'centering': 'Centering',
    'corners':   'Corners',
    'edges':     'Edges',
    'surface':   'Surface',
    'other':     'Other',
}


def _print_side_issues(issues: dict) -> bool:
    """Print per-side issues dict; returns True if any issues found."""
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'      {cat.upper()}')
            for item in items:
                print(f'        • {item}')
    return has_any


def print_result(result: dict, label: str = '') -> None:
    """Pretty-print a single grading result to stdout."""
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    grade  = result.get('grade_estimate', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    cv     = result.get('_cv')
    usage  = result.get('_usage', {})
    mode   = result.get('analysis_mode', 'front_only')

    sep = '=' * 65
    print(f'\n{sep}')
    if label:
        print(f'  {label}')
    print(sep)

    # Card identity
    name    = card.get('name') or '(unknown)'
    cset    = card.get('set') or '?'
    year    = card.get('year') or '?'
    num     = card.get('number') or '?'
    id_conf = card.get('confidence', '?')
    print(f'\n🃏  {name}  [{cset} {year} #{num}]')
    print(f'    ID confidence: {id_conf}   Mode: {mode}')

    # Image quality
    status = iq.get('status', '?').upper()
    print(f'\n📷  Image Quality: {status}')
    chip_defs = [
        ('front_present', 'Front'), ('back_present', 'Back'),
        ('centering_visible', 'Centering'), ('corners_visible', 'Corners'),
        ('edges_visible', 'Edges'),         ('surface_visible', 'Surface'),
    ]
    chips = '  '.join(f"{'✓' if iq.get(k) else '✗'} {lbl}" for k, lbl in chip_defs)
    print(f'    {chips}')
    for w in iq.get('warnings', []):
        print(f'    ⚠ {w}')

    # CV measurements
    if cv:
        bf = '🔴' if cv['is_blurry'] else '🟢'
        gf = '🔴' if cv['has_glare'] else '🟢'
        print(f'\n🔬  CV: blur {cv["blur_score"]:.1f}{bf}  '
              f'glare {cv["glare_fraction"]*100:.1f}%{gf}  '
              f'brightness {cv["brightness_mean"]:.0f}')

    # ── Front Analysis ────────────────────────────────────────────
    print(f'\n── Front Analysis ──────────────────────────────────────────')
    if front.get('assessable') is False:
        print('    ✗ Not assessable — no front image provided')
    else:
        centering = front.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        front_issues = front.get('issues', {})
        if not _print_side_issues({k: v for k, v in front_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Back Analysis ─────────────────────────────────────────────
    print(f'\n── Back Analysis ───────────────────────────────────────────')
    if back.get('assessable') is False:
        print('    ✗ Not assessable — no back image provided')
    else:
        centering = back.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        back_issues = back.get('issues', {})
        if not _print_side_issues({k: v for k, v in back_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Grade estimate ────────────────────────────────────────────
    gr   = grade.get('grade_range', '?')
    conf = grade.get('confidence', '?')
    lf   = grade.get('limiting_factor')
    lf_str = f'  [limiting: {lf}]' if lf else ''
    print(f'\n📊  Grade Estimate: {gr}  ({conf} confidence){lf_str}')
    dist = grade.get('distribution', {})
    for g in range(10, 0, -1):
        prob = dist.get(str(g), 0)
        bar  = '█' * int(prob * 32)
        print(f'    PSA {g:2d} │ {bar:<32} {prob*100:5.1f}%')

    # ── Grading decision ──────────────────────────────────────────
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '❓')
    print(f'\n{emoji}  Grading Candidate: {gdc.upper()}')
    print(f'    {gd.get("reason", "")}')

    caveats = gd.get('caveats', [])
    if caveats:
        print(f'\n⛔  Caveats:')
        for c in caveats:
            print(f'    • {c}')

    # ── Combined issues ───────────────────────────────────────────
    print('\n⚠️  Combined Issues (worst-case):')
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'    {cat.upper()}')
            for item in items:
                print(f'      • {item}')
    if not has_any:
        print('    ✓ No significant issues detected')

    # Token usage
    if usage:
        print(f'\n💰  Tokens: {usage.get("input_tokens",0):,} in '
              f'/ {usage.get("output_tokens",0):,} out')
    print(f'\n{sep}\n')


def plot_result(result: dict, label: str = '') -> None:
    """Matplotlib visualisation of a single grading result."""
    grade  = result.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    mode   = result.get('analysis_mode', '?')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: probability bars ────────────────────────────────────
    ax1 = axes[0]
    grades = list(range(1, 11))
    probs  = [dist.get(str(g), 0) * 100 for g in grades]
    colors = [GRADE_COLORS[g] for g in grades]
    bars   = ax1.bar(grades, probs, color=colors, width=0.7, edgecolor='#30363d')
    ax1.set_xlabel('PSA Grade')
    ax1.set_ylabel('Probability (%)')
    ax1.set_title(
        f'{grade.get("grade_range","?")}  ({grade.get("confidence","?")} confidence)',
        color='white')
    ax1.set_xticks(grades)
    ax1.set_ylim(0, max(probs) * 1.15 + 1)
    ax1.spines['bottom'].set_color('#30363d')
    ax1.spines['left'].set_color('#30363d')
    for bar, prob in zip(bars, probs):
        if prob > 2:
            ax1.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.4,
                     f'{prob:.0f}%', ha='center', va='bottom',
                     color='white', fontsize=8)

    # ── Right: summary text ───────────────────────────────────────
    ax2 = axes[1]
    ax2.axis('off')
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '?')

    # Build front/back summary strings
    def side_summary(side: dict, name: str) -> list[str]:
        lines = [f'{name}:']
        if not side.get('assessable', True):
            lines.append('  ✗ not assessable')
            return lines
        c = side.get('centering')
        if c:
            lines.append(f'  centering {c}')
        side_issues = side.get('issues', {})
        count = sum(len(v) for v in side_issues.values() if isinstance(v, list))
        lines.append(f'  {count} issue(s) detected')
        return lines

    lines = [
        f"Card: {card.get('name') or 'Unknown'}",
        f"Set:  {card.get('set') or '?'}  {card.get('year') or ''}",
        f"ID confidence: {card.get('confidence', '?')}",
        f"Mode: {mode}   IQ: {iq.get('status','?')}",
        "",
    ]
    lines += side_summary(front, 'Front')
    lines += side_summary(back, 'Back')
    lines += [
        "",
        f"{emoji} Grading: {gdc.upper()}",
        textwrap.fill(gd.get('reason', ''), 42),
    ]
    caveats = gd.get('caveats', [])
    if caveats:
        lines += ["", "⛔ Caveats:"]
        for c in caveats:
            lines.append(f"  • {textwrap.shorten(c, 40)}")
    lines += ["", "Combined Issues:"]
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            lines.append(f"  {cat.upper()} ({len(items)})")
            for it in items:
                lines.append(f"    • {textwrap.shorten(it, 36)}")
    if not any(issues.get(c) for c in ['centering','corners','edges','surface','other']):
        lines.append('  ✓ No significant issues')

    ax2.text(0.04, 0.97, '\n'.join(lines),
             transform=ax2.transAxes, color='white',
             fontsize=9.5, va='top', family='monospace', linespacing=1.55)
    ax2.set_title(label or 'Assessment', color='white')

    plt.suptitle(label, color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


print('✅ Display helpers ready')

---
## Panel-style Centering Inspection

Mirrors what the Chrome extension's side panel renders when the user clicks **Centering** in the lightbox.

- **Outer card boundary** (blue thin stroke) — from `detect_card_bounds()` (Python port of `cvDetectors.ts::detectCardBounds`)
- **Inner printed frame** (yellow thin stroke) — from `detect_inner_frame()` (Python port of `cvDetectors.ts::detectInnerFrame`)
- **T/B/L/R margin labels** — derived percentages, sum-to-100 per axis (matches grader "53/47" convention)
- **Interpretation** — well-centered / slightly off / noticeably off / severely off
- **Claude's read** — verbatim from `result['front_analysis']['centering']`

For borderless / full-art cards (where inner-frame detection returns `None`), the visualization gracefully degrades to outer-only with an explanatory message — same behavior as the panel.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Python ports of cvDetectors.ts (centering subsystem)
#  Kept in sync with src/lib/grading/cvDetectors.ts
# ══════════════════════════════════════════════════════════════════════════════

from PIL import Image, ImageDraw, ImageFont
import matplotlib.patches as mpatches

def _load_rgb(image_path_or_url: str) -> np.ndarray | None:
    """Load an image (local path or URL) as an RGB numpy array."""
    try:
        if image_path_or_url.startswith(('http://', 'https://')):
            resp = requests.get(image_path_or_url, timeout=15)
            resp.raise_for_status()
            pil = Image.open(io.BytesIO(resp.content)).convert('RGB')
        else:
            pil = Image.open(image_path_or_url).convert('RGB')
        return np.array(pil)
    except Exception as exc:
        print(f'  failed to load {image_path_or_url}: {exc}')
        return None


def detect_card_bounds(img_rgb: np.ndarray) -> dict | None:
    """
    Python port of cvDetectors.ts::detectCardBounds.

    Returns {'left','top','width','height'} in original-image pixel coords,
    or None when detection is unreliable.
    """
    DETECT_MAX      = 256
    CORNER_SAMPLE   = 12
    RGB_DIST_THRESH = 35
    MARGIN_FRAC     = 0.08
    MIN_FG_FRAC     = 0.05
    MIN_CROP_FRAC   = 0.25
    MAX_CROP_FRAC   = 0.98   # matches the raised cap in cvDetectors.ts

    orig_h, orig_w = img_rgb.shape[:2]
    if not orig_w or not orig_h:
        return None

    # Downsample while preserving aspect ratio
    scale = DETECT_MAX / max(orig_w, orig_h)
    if scale < 1.0:
        small = cv2.resize(img_rgb, (int(orig_w * scale), int(orig_h * scale)),
                            interpolation=cv2.INTER_AREA)
    else:
        small = img_rgb
    sH, sW = small.shape[:2]

    # Median R/G/B from each of the four CORNER_SAMPLE × CORNER_SAMPLE patches
    def median_patch(r0, c0):
        patch = small[r0:r0+CORNER_SAMPLE, c0:c0+CORNER_SAMPLE].reshape(-1, 3)
        return np.median(patch, axis=0)

    patches = [
        median_patch(0,                  0),
        median_patch(0,                  sW - CORNER_SAMPLE),
        median_patch(sH - CORNER_SAMPLE, 0),
        median_patch(sH - CORNER_SAMPLE, sW - CORNER_SAMPLE),
    ]
    bg = np.round(np.mean(patches, axis=0)).astype(np.int32)

    diff = small.astype(np.int32) - bg.reshape(1, 1, 3)
    dist = np.sqrt(np.sum(diff * diff, axis=2))
    fg   = dist > RGB_DIST_THRESH

    fg_count = int(fg.sum())
    if fg_count / (sW * sH) < MIN_FG_FRAC:
        return None

    rows_with_fg = np.where(fg.any(axis=1))[0]
    cols_with_fg = np.where(fg.any(axis=0))[0]
    if rows_with_fg.size == 0 or cols_with_fg.size == 0:
        return None

    min_row, max_row = int(rows_with_fg[0]), int(rows_with_fg[-1])
    min_col, max_col = int(cols_with_fg[0]), int(cols_with_fg[-1])

    # Expand 8% for the card's white border
    m_row = round((max_row - min_row) * MARGIN_FRAC)
    m_col = round((max_col - min_col) * MARGIN_FRAC)
    e_min_row = max(0,      min_row - m_row)
    e_max_row = min(sH - 1, max_row + m_row)
    e_min_col = max(0,      min_col - m_col)
    e_max_col = min(sW - 1, max_col + m_col)

    # Scale back to original image coordinates
    scale_x = orig_w / sW
    scale_y = orig_h / sH
    left   = max(0,        round(e_min_col * scale_x))
    top    = max(0,        round(e_min_row * scale_y))
    right  = min(orig_w,   round(e_max_col * scale_x))
    bottom = min(orig_h,   round(e_max_row * scale_y))
    cropW  = right - left
    cropH  = bottom - top

    if cropW <= 0 or cropH <= 0:
        return None

    crop_frac = (cropW * cropH) / (orig_w * orig_h)
    if crop_frac < MIN_CROP_FRAC: return None
    if crop_frac > MAX_CROP_FRAC: return None

    return {'left': left, 'top': top, 'width': cropW, 'height': cropH}


def detect_inner_frame(card_only_rgb: np.ndarray) -> dict | None:
    """
    Python port of cvDetectors.ts::detectInnerFrame.

    Sharp-only Sobel-style gradient scan inward from each side. Returns
    {'x','y','w','h','confidence'} in card-fraction space (0–1), or None
    for borderless / full-art / low-contrast cards.
    """
    DETECT_MAX     = 384
    SEARCH_FRAC    = 0.30
    MIN_INSET_FRAC = 0.015
    MAX_INSET_FRAC = 0.22
    PEAK_THRESH    = 0.45

    H0, W0 = card_only_rgb.shape[:2]
    scale  = DETECT_MAX / max(W0, H0) if max(W0, H0) > DETECT_MAX else 1.0
    if scale < 1.0:
        small = cv2.resize(card_only_rgb, (int(W0*scale), int(H0*scale)),
                            interpolation=cv2.INTER_AREA)
    else:
        small = card_only_rgb

    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W = gray.shape
    if W < 40 or H < 40:
        return None

    # Horizontal gradient column profile (tall peaks at vertical frame edges)
    # gradH[r, c] = |gray[r, c+1] - gray[r, c-1]|
    gradH = np.abs(gray[:, 2:] - gray[:, :-2])
    colGrad = np.zeros(W, dtype=np.float64)
    colGrad[1:W-1] = gradH.sum(axis=0)

    # Vertical gradient row profile (tall peaks at horizontal frame edges)
    gradV = np.abs(gray[2:, :] - gray[:-2, :])
    rowGrad = np.zeros(H, dtype=np.float64)
    rowGrad[1:H-1] = gradV.sum(axis=1)

    def find_inner_edge(profile, length, from_start):
        search_max = int(length * SEARCH_FRAC)
        min_inset  = int(length * MIN_INSET_FRAC)
        max_inset  = int(length * MAX_INSET_FRAC)
        max_val    = float(profile.max())
        if max_val <= 0:
            return None
        abs_thresh = max_val * PEAK_THRESH

        best_idx, best_strength = -1, 0.0
        lo, hi = min_inset, min(max_inset, search_max)
        for i in range(lo, hi + 1):
            idx = i if from_start else (length - 1 - i)
            v = profile[idx]
            if v >= abs_thresh and v > best_strength:
                # local max within ±2 window
                i0 = max(0, idx - 2)
                i1 = min(length - 1, idx + 2)
                if v >= profile[i0:i1+1].max():
                    best_strength, best_idx = v, idx
        if best_idx < 0:
            return None
        return {'idx': best_idx, 'strength': best_strength / max_val}

    left   = find_inner_edge(colGrad, W, True)
    right  = find_inner_edge(colGrad, W, False)
    top    = find_inner_edge(rowGrad, H, True)
    bottom = find_inner_edge(rowGrad, H, False)
    if None in (left, right, top, bottom):
        return None
    if right['idx'] <= left['idx'] + 8 or bottom['idx'] <= top['idx'] + 8:
        return None

    avg_strength = (left['strength'] + right['strength']
                    + top['strength'] + bottom['strength']) / 4
    confidence = ('high'   if avg_strength >= 0.75 else
                  'medium' if avg_strength >= 0.55 else 'low')

    return {
        'x': left['idx']   / W,
        'y': top['idx']    / H,
        'w': (right['idx']  - left['idx']) / W,
        'h': (bottom['idx'] - top['idx'])  / H,
        'confidence': confidence,
    }


CENTERING_INTERPRETATION_TEXT = {
    'well_centered':  'Well-centered',
    'slightly_off':   'Slightly off-center',
    'noticeably_off': 'Noticeably off-center',
    'severely_off':   'Severely off-center',
    'unavailable':    'Centering measurement unavailable',
}


def build_centering_measurement(card_bounds_pct: dict, inner_frame: dict | None) -> dict:
    """Python port of cvDetectors.ts::buildCenteringMeasurement."""
    if inner_frame is None:
        return {
            'card_bbox_pct':        card_bounds_pct,
            'inner_frame_bbox_pct': None,
            'margins_pct':          None,
            'ratios':               None,
            'interpretation':       'unavailable',
            'confidence':           'low',
            'fallback_reason':      'borderless_card',
        }
    left_raw   = inner_frame['x']
    right_raw  = 1 - (inner_frame['x'] + inner_frame['w'])
    top_raw    = inner_frame['y']
    bottom_raw = 1 - (inner_frame['y'] + inner_frame['h'])
    horiz_sum  = left_raw + right_raw
    vert_sum   = top_raw  + bottom_raw
    if horiz_sum <= 1e-6 or vert_sum <= 1e-6:
        return {
            'card_bbox_pct':        card_bounds_pct,
            'inner_frame_bbox_pct': inner_frame,
            'margins_pct':          None,
            'ratios':               None,
            'interpretation':       'unavailable',
            'confidence':           inner_frame['confidence'],
            'fallback_reason':      'low_contrast',
        }
    left_pct   = round((left_raw  / horiz_sum) * 100)
    right_pct  = 100 - left_pct
    top_pct    = round((top_raw   / vert_sum)  * 100)
    bottom_pct = 100 - top_pct
    worst = max(abs(50 - left_pct), abs(50 - top_pct))
    interpretation = ('well_centered'   if worst <= 2  else
                      'slightly_off'    if worst <= 5  else
                      'noticeably_off'  if worst <= 10 else
                      'severely_off')
    return {
        'card_bbox_pct':        card_bounds_pct,
        'inner_frame_bbox_pct': inner_frame,
        'margins_pct':          {'left': left_pct, 'right': right_pct,
                                  'top': top_pct,  'bottom': bottom_pct},
        'ratios':               {'left_right': f'{left_pct}/{right_pct}',
                                  'top_bottom': f'{top_pct}/{bottom_pct}'},
        'interpretation':       interpretation,
        'confidence':           inner_frame['confidence'],
        'fallback_reason':      None,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  Panel-style centering visualization
#  Mirrors extension/sidepanel.js::buildCenteringSVG + renderCenteringBanner
# ══════════════════════════════════════════════════════════════════════════════

def plot_centering_inspection(image_path_or_url: str,
                               claude_note: str | None = None,
                               figsize: tuple = (8, 11)) -> dict | None:
    """
    Render the same centering inspection view the Chrome extension's side
    panel shows when the user clicks Centering in the lightbox.

    Computes (or accepts via the returned dict) a centering measurement,
    then plots the cropped card with:
      - blue outer rectangle  (outer card edge)
      - yellow inner rectangle (inner printed frame) when detected
      - T/B/L/R margin labels  in the gaps between rectangles
    Below the image: ratio summary, interpretation, and Claude's read.

    Returns the centering measurement dict (or None when the card couldn't
    be isolated from the photo).
    """
    img = _load_rgb(image_path_or_url)
    if img is None:
        print('  cannot load image')
        return None

    H0, W0 = img.shape[:2]
    bounds = detect_card_bounds(img)

    fig = plt.figure(figsize=figsize)
    # Layout: image on top (75% height), banner block on bottom (25%)
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.08)
    ax_img    = fig.add_subplot(gs[0])
    ax_banner = fig.add_subplot(gs[1])
    ax_img.set_facecolor('#0d1117')
    ax_banner.set_facecolor('#0d1117')
    fig.patch.set_facecolor('#0d1117')

    if bounds is None:
        # Fallback: show the full photo with a "Card fills frame" banner
        ax_img.imshow(img)
        ax_img.set_title('Card fills frame — CV measurement unavailable',
                         color='#cfe3ff', fontsize=11)
        ax_img.axis('off')
        _render_centering_banner(ax_banner, None, claude_note)
        plt.show()
        return None

    # Crop to card region only — this is what the panel SVG slot covers
    L, T, BW, BH = bounds['left'], bounds['top'], bounds['width'], bounds['height']
    card_only = img[T:T+BH, L:L+BW]

    # Inner frame detection (in card-fraction space)
    inner = detect_inner_frame(card_only)
    card_bounds_pct = {
        'x': L / W0, 'y': T / H0, 'w': BW / W0, 'h': BH / H0,
    }
    measurement = build_centering_measurement(card_bounds_pct, inner)

    # Render the cropped card with overlays
    ax_img.imshow(card_only, extent=[0, 100, 100, 0])  # viewBox 0–100, y inverted
    ax_img.set_xlim(0, 100); ax_img.set_ylim(100, 0)
    ax_img.set_aspect(BW / BH * 1.0)
    ax_img.axis('off')

    # Outer card boundary — thin blue rect at viewBox edges (inset 0.4)
    outer_rect = mpatches.Rectangle(
        (0.4, 0.4), 99.2, 99.2,
        linewidth=1.4, edgecolor='#5fa8ff', facecolor='none',
    )
    ax_img.add_patch(outer_rect)

    # Inner frame + labels (only when detected)
    if inner is not None:
        ix, iy = inner['x'] * 100, inner['y'] * 100
        iw, ih = inner['w'] * 100, inner['h'] * 100
        inner_rect = mpatches.Rectangle(
            (ix, iy), iw, ih,
            linewidth=1.4, edgecolor='#ffd84a', facecolor='none',
        )
        ax_img.add_patch(inner_rect)

        m = measurement['margins_pct']
        if m is not None:
            label_defs = [
                (f"T: {m['top']}",    ix + iw/2,                  iy / 2),
                (f"B: {m['bottom']}", ix + iw/2,                  iy + ih + (100 - iy - ih) / 2),
                (f"L: {m['left']}",   ix / 2,                     iy + ih/2),
                (f"R: {m['right']}",  ix + iw + (100 - ix - iw)/2, iy + ih/2),
            ]
            for text, x, y in label_defs:
                ax_img.text(x, y, text,
                            ha='center', va='center',
                            color='white', fontsize=10, fontweight='bold',
                            path_effects=[
                                _stroke_effect(linewidth=2.5, color=(0, 0, 0, 0.78))
                            ])

    _render_centering_banner(ax_banner, measurement, claude_note)
    plt.tight_layout()
    plt.show()
    return measurement


def _stroke_effect(linewidth=2.5, color=(0, 0, 0, 0.78)):
    """matplotlib text stroke for legibility on any background."""
    from matplotlib import patheffects
    return patheffects.withStroke(linewidth=linewidth, foreground=color)


def _render_centering_banner(ax, measurement: dict | None, claude_note: str | None):
    """Mirrors renderCenteringBanner from sidepanel.js — stacked layout."""
    import textwrap as _tw
    ax.axis('off')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    banner_bg = mpatches.FancyBboxPatch(
        (0.005, 0.02), 0.99, 0.96,
        boxstyle='round,pad=0.012,rounding_size=0.02',
        linewidth=1, edgecolor='#5fa8ff47', facecolor='#5fa8ff1a',
        transform=ax.transAxes,
    )
    ax.add_patch(banner_bg)

    if measurement is None:
        headline       = 'Card fills frame'
        interpretation = 'CV measurement unavailable for tightly-cropped photos'
        helper         = "Visual overlay disabled — see Claude's read below."
    elif measurement['inner_frame_bbox_pct'] is None or measurement['margins_pct'] is None:
        headline       = 'Outer card detected'
        interpretation = CENTERING_INTERPRETATION_TEXT.get(
            measurement['interpretation'], 'Centering measurement unavailable')
        helper         = ('Inner frame not detected — common for full-art / borderless cards.'
                          if measurement.get('fallback_reason') == 'borderless_card'
                          else 'Inner frame could not be measured reliably for this image.')
    else:
        r = measurement['ratios']
        headline       = f"{r['top_bottom']} T/B  ·  {r['left_right']} L/R"
        interpretation = CENTERING_INTERPRETATION_TEXT.get(
            measurement['interpretation'], '')
        helper         = 'Measured from outer card edge to inner print frame.'

    # Stacked layout — each line on its own row, no horizontal collisions
    # Row 1: "● Centering   <headline>"  (headline starts after enough padding)
    ax.text(0.03, 0.88, '● Centering',
            transform=ax.transAxes, color='#cfe3ff',
            fontsize=10, fontweight='bold', va='top')
    ax.text(0.97, 0.88, headline,
            transform=ax.transAxes, color='white',
            fontsize=12, fontweight='bold', va='top', ha='right',
            family='monospace')

    # Row 2: interpretation
    ax.text(0.03, 0.66, interpretation,
            transform=ax.transAxes, color='white',
            fontsize=11, va='top')

    # Row 3: Claude's read (wrapped) — only when present
    next_y = 0.46
    if claude_note:
        wrapped = _tw.fill(claude_note, width=60)
        ax.text(0.03, next_y, "Claude's read:",
                transform=ax.transAxes, color='#8b949e',
                fontsize=9, fontweight='bold', va='top')
        ax.text(0.03, next_y - 0.10, wrapped,
                transform=ax.transAxes, color='white',
                fontsize=9, va='top')
        next_y -= 0.10 + 0.06 * (wrapped.count(chr(10)) + 1)

    # Row 4: helper
    ax.text(0.03, max(next_y, 0.08), helper,
            transform=ax.transAxes, color='#8b949e',
            fontsize=9, style='italic', va='top')


def plot_centering_inspection_from_result(result: dict,
                                           image_path_or_url: str,
                                           figsize: tuple = (8, 11)) -> dict | None:
    """
    Convenience: pull Claude's textual centering note out of `result` and
    call plot_centering_inspection. Use this right after `result = grade_with_claude(...)`
    to render the panel-style centering view for the front image.
    """
    front = result.get('front_analysis', {})
    claude_note = front.get('centering')
    return plot_centering_inspection(image_path_or_url, claude_note=claude_note, figsize=figsize)


print('✅ Centering inspection helpers ready')
print('   • detect_card_bounds()             — port of cvDetectors.ts')
print('   • detect_inner_frame()             — port of cvDetectors.ts')
print('   • build_centering_measurement()    — port of cvDetectors.ts')
print('   • plot_centering_inspection(path)  — panel-style visualization')
print('   • plot_centering_inspection_from_result(result, path)')


---
## YOLO-based Card Crop

Robust card detection using YOLOv8 segmentation. Unlike the color-threshold
`detect_card_bounds()`, this approach:

- Handles **busy / textured / shadowed backgrounds** that confuse the corner-patch
  background estimate
- Handles **partial occlusion** (hands, holders, sleeves) by relying on shape, not
  pixel similarity
- Returns both an **axis-aligned crop** and a **perspective-rectified card** by
  warping the detected 4-corner quadrilateral to a canonical aspect ratio

YOLOv8 nano-seg is COCO-trained (no "card" class), so we don't rely on the class
label — instead we score every segmentation mask for *card-likeness*:

| Signal | Card expectation |
|---|---|
| 4-corner polygon | `cv2.approxPolyDP` reduces to ≤ 4 vertices |
| Aspect ratio | short/long ≈ 0.65–0.78 (real card is 2.5″×3.5″ ≈ 0.714) |
| Convex / rectangular | corner angles within ±30° of 90° |
| Coverage | bbox area ≥ 5% of the image |

The model file `yolov8n-seg.pt` is already in the notebook directory (~7 MB).
Requires `ultralytics` (already installed if the centering cells worked).

For a custom-trained card detector, just point `yolo_crop_card(..., model_path=...)`
at your weights — same interface, same return shape.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  YOLO-based card crop — handles textured backgrounds + perspective skew
# ══════════════════════════════════════════════════════════════════════════════

from pathlib import Path as _Path
import contextlib as _ctx

# Lazy import so the rest of the notebook stays usable if ultralytics is absent
try:
    from ultralytics import YOLO as _YOLO
    _ULTRA_AVAILABLE = True
except ImportError:
    _ULTRA_AVAILABLE = False

# Cache the loaded model across calls (loading is the slow part)
_YOLO_CACHE = {}

# Real Pokémon cards: 2.5" × 3.5" → short/long ratio = 0.7143
CARD_ASPECT_RATIO = 2.5 / 3.5         # ≈ 0.714
CARD_ASPECT_TOL   = 0.10               # accept 0.61 – 0.81
CANONICAL_CARD_W  = 500                # output rectified width
CANONICAL_CARD_H  = 700                # output rectified height (matches 5:7)

def _get_yolo(model_path: str):
    """Load + cache the YOLO model. Silences ultralytics startup chatter."""
    if not _ULTRA_AVAILABLE:
        raise RuntimeError('ultralytics not installed — `pip install ultralytics`')
    if model_path not in _YOLO_CACHE:
        with _ctx.redirect_stdout(io.StringIO()):
            _YOLO_CACHE[model_path] = _YOLO(model_path)
    return _YOLO_CACHE[model_path]


def _order_quad(pts: np.ndarray) -> np.ndarray:
    """Order a 4-point quad as [TL, TR, BR, BL] regardless of input order."""
    pts = pts.reshape(4, 2).astype(np.float32)
    s    = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).flatten()
    return np.array([
        pts[np.argmin(s)],     # TL: smallest x+y
        pts[np.argmin(diff)],  # TR: smallest y-x
        pts[np.argmax(s)],     # BR: largest x+y
        pts[np.argmax(diff)],  # BL: largest y-x
    ], dtype=np.float32)


def _corner_angles(quad_tl_tr_br_bl: np.ndarray) -> np.ndarray:
    """Angle at each corner (degrees) for an ordered quad."""
    q = quad_tl_tr_br_bl
    angles = []
    for i in range(4):
        prev_pt = q[(i - 1) % 4]
        curr_pt = q[i]
        next_pt = q[(i + 1) % 4]
        v1 = prev_pt - curr_pt
        v2 = next_pt - curr_pt
        cos_a = float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9))
        angles.append(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
    return np.array(angles)


def _card_likeness_score(quad: np.ndarray, mask_area: int, img_area: int) -> dict:
    """
    Score how 'card-like' a 4-point polygon is. Higher is better.

    Returns dict with score + breakdown + `valid: bool`. valid=False marks
    candidates that fail HARD rejection rules:
      - aspect ratio outside [0.55, 0.85]   → not card-shaped
      - coverage > 0.92                       → likely the photo bounds, not a card
      - coverage < 0.05                       → too small to be the subject
      - any corner angle > 30° off square     → not a quad
    """
    q = _order_quad(quad)
    edges = np.array([np.linalg.norm(q[(i + 1) % 4] - q[i]) for i in range(4)])
    short_pair = sorted([(edges[0] + edges[2]) / 2, (edges[1] + edges[3]) / 2])
    short, long_ = short_pair
    aspect = short / max(long_, 1e-6)
    aspect_dev = abs(aspect - CARD_ASPECT_RATIO)
    aspect_score = max(0.0, 1.0 - aspect_dev / CARD_ASPECT_TOL)

    angles = _corner_angles(q)
    angle_dev = float(np.mean(np.abs(angles - 90)))
    angle_score = max(0.0, 1.0 - angle_dev / 30.0)
    max_angle_dev = float(np.max(np.abs(angles - 90)))

    coverage = mask_area / max(img_area, 1)
    coverage_score = min(1.0, coverage / 0.10)

    # Hard rejection — any one of these vetoes the candidate
    aspect_ok   = 0.55 <= aspect <= 0.85
    coverage_ok = 0.05 <= coverage <= 0.92
    angle_ok    = max_angle_dev <= 30.0
    valid       = aspect_ok and coverage_ok and angle_ok

    # Aspect dominates the composite — penalize harder than before
    score = 0.55 * aspect_score + 0.30 * angle_score + 0.15 * coverage_score

    return {
        'score': round(score, 4),
        'valid': valid,
        'aspect': round(float(aspect), 3),
        'aspect_score': round(aspect_score, 3),
        'aspect_ok': aspect_ok,
        'angle_dev': round(angle_dev, 1),
        'max_angle_dev': round(max_angle_dev, 1),
        'angle_score': round(angle_score, 3),
        'angle_ok': angle_ok,
        'coverage': round(coverage, 4),
        'coverage_score': round(coverage_score, 3),
        'coverage_ok': coverage_ok,
    }


def _mask_to_quad(mask: np.ndarray, epsilon_frac: float = 0.02) -> np.ndarray | None:
    """
    Reduce a boolean segmentation mask to a 4-point polygon.

    Pipeline: largest external contour → approxPolyDP with progressive epsilon
    until exactly 4 vertices, falling back to minAreaRect when no clean 4-poly.
    """
    mask_u8 = (mask.astype(np.uint8)) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(cnt, True)

    # Try progressively coarser approximations until we get 4 vertices
    for eps_mult in (0.01, 0.015, 0.02, 0.03, 0.04, 0.06, 0.08):
        approx = cv2.approxPolyDP(cnt, eps_mult * peri, True)
        if len(approx) == 4:
            return approx.reshape(4, 2).astype(np.float32)

    # Fallback: rotated minimum-area rectangle
    rect = cv2.minAreaRect(cnt)
    box  = cv2.boxPoints(rect)
    return box.astype(np.float32)


def _perspective_warp(img_rgb: np.ndarray, quad: np.ndarray,
                       out_w: int = CANONICAL_CARD_W,
                       out_h: int = CANONICAL_CARD_H) -> np.ndarray:
    """Warp a 4-point quad to a canonical out_w × out_h rectangle."""
    q = _order_quad(quad)
    dst = np.array([[0, 0], [out_w - 1, 0],
                    [out_w - 1, out_h - 1], [0, out_h - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(q, dst)
    return cv2.warpPerspective(img_rgb, M, (out_w, out_h))


def _axis_aligned_crop(img_rgb: np.ndarray, quad: np.ndarray) -> tuple[np.ndarray, dict]:
    """Tight axis-aligned bbox crop around the quad."""
    q = _order_quad(quad)
    x0, y0 = q.min(axis=0).astype(int)
    x1, y1 = q.max(axis=0).astype(int)
    H, W = img_rgb.shape[:2]
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(W, x1), min(H, y1)
    return img_rgb[y0:y1, x0:x1], {'left': x0, 'top': y0, 'width': x1 - x0, 'height': y1 - y0}


def _opencv_fallback_quad(img_rgb: np.ndarray, debug: bool = False) -> np.ndarray | None:
    """
    OpenCV-only quad detection — robust to holographic noise and PSA slabs.

    Strategy:
      1. Bilateral filter (preserves edges, smooths interior holographic glare)
      2. Canny on multiple sources (grayscale + saturation + value)
      3. Morphological close to connect broken edges
      4. Enumerate up to TOP_CONTOURS by area, approximate each to a polygon
      5. SCORE every valid 4-vertex candidate using _card_likeness_score
      6. Return the highest-scoring valid candidate (NOT just the largest)

    This handles graded-slab photos where the slab outer edge is the largest
    contour but the actual card is a smaller rectangle inside the slab.
    """
    H, W = img_rgb.shape[:2]
    img_area = H * W

    # Bilateral filter — slow but kills holographic noise without blurring card edges
    smoothed = cv2.bilateralFilter(img_rgb, d=9, sigmaColor=75, sigmaSpace=75)

    gray = cv2.cvtColor(smoothed, cv2.COLOR_RGB2GRAY)
    hsv  = cv2.cvtColor(smoothed, cv2.COLOR_RGB2HSV)
    sat  = hsv[..., 1]
    val  = hsv[..., 2]

    TOP_CONTOURS = 30
    best_quad, best_score = None, 0.0
    best_scoring = None

    for src_name, channel in (('gray', gray), ('sat', sat), ('val', val)):
        for canny_low, canny_high in ((40, 120), (60, 180)):
            edges = cv2.Canny(channel, canny_low, canny_high)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
            closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
            contours, _ = cv2.findContours(closed, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in sorted(contours, key=cv2.contourArea, reverse=True)[:TOP_CONTOURS]:
                area = cv2.contourArea(cnt)
                if area < 0.04 * img_area or area > 0.92 * img_area:
                    continue
                peri = cv2.arcLength(cnt, True)
                # Try several epsilon values to land on a 4-vertex polygon
                for eps_mult in (0.01, 0.015, 0.02, 0.03, 0.04, 0.06):
                    approx = cv2.approxPolyDP(cnt, eps_mult * peri, True)
                    if len(approx) != 4:
                        continue
                    quad = approx.reshape(4, 2).astype(np.float32)
                    scoring = _card_likeness_score(quad, int(area), img_area)
                    if scoring['valid'] and scoring['score'] > best_score:
                        best_score = scoring['score']
                        best_quad  = quad
                        best_scoring = scoring
                        if debug:
                            print(f'    [{src_name},canny={canny_low}-{canny_high},eps={eps_mult}] '
                                  f'score={scoring["score"]} aspect={scoring["aspect"]} '
                                  f'cov={scoring["coverage"]}')
                    break  # this contour got a 4-vertex approximation, don't try other epsilons

    if debug and best_scoring:
        print(f'  best fallback quad: score={best_score:.3f} '
              f'aspect={best_scoring["aspect"]} '
              f'coverage={best_scoring["coverage"]}')

    return best_quad


def yolo_crop_card(image,
                    model_path: str = 'yolov8n-seg.pt',
                    confidence_threshold: float = 0.15,
                    canonical_size: tuple = (CANONICAL_CARD_W, CANONICAL_CARD_H),
                    debug: bool = False) -> dict:
    """
    Detect and crop the rectangular card from any image.

    Returns:
        dict with:
            rectified     : np.ndarray (H, W, 3) — perspective-warped to canonical_size
            axis_aligned  : np.ndarray            — tight bbox crop, no warp
            bbox          : dict                  — {left, top, width, height} in original coords
            quad          : np.ndarray (4, 2)     — corners in original-image px, ordered TL/TR/BR/BL
            method        : str                   — 'yolo' | 'opencv-fallback'
            detected_class: str | None            — COCO class that fired (yolo path only)
            score         : float                 — card-likeness score (0–1)
            scoring       : dict                  — full scoring breakdown
            mask          : np.ndarray | None     — boolean mask (yolo path only)

    Returns None when no card-like quadrilateral could be found.
    """
    # Normalize input → RGB ndarray
    if isinstance(image, (str, _Path)):
        img_rgb = _load_rgb(str(image))
        if img_rgb is None:
            return None
    elif isinstance(image, np.ndarray):
        img_rgb = image
    else:
        raise TypeError(f'image must be path or ndarray, got {type(image)}')

    H, W = img_rgb.shape[:2]
    img_area = H * W

    candidates = []   # (quad, mask, score_dict, class_name)

    # ── Path A: YOLO segmentation ──
    if _ULTRA_AVAILABLE:
        try:
            model = _get_yolo(model_path)
            with _ctx.redirect_stdout(io.StringIO()):
                results = model(img_rgb[..., ::-1] if False else img_rgb,   # YOLO accepts RGB
                                conf=confidence_threshold, verbose=False)
            for r in results:
                if r.masks is None:
                    continue
                # r.masks.data: (N, mh, mw) at model resolution; we want full-resolution
                masks_np = r.masks.data.cpu().numpy()           # (N, mh, mw) float 0–1
                classes  = r.boxes.cls.cpu().numpy().astype(int)
                for i, m_low in enumerate(masks_np):
                    # Resize mask to original resolution
                    m_full = cv2.resize(m_low, (W, H), interpolation=cv2.INTER_NEAREST)
                    m_bool = m_full > 0.5
                    if m_bool.sum() < 0.02 * img_area:
                        continue
                    quad = _mask_to_quad(m_bool)
                    if quad is None:
                        continue
                    scoring = _card_likeness_score(quad, int(m_bool.sum()), img_area)
                    cls_name = model.names.get(int(classes[i]), '?')
                    candidates.append((quad, m_bool, scoring, cls_name))
        except Exception as exc:
            if debug:
                print(f'  YOLO inference failed: {exc}')

    # Pick the best-scoring YOLO candidate
    # Only consider candidates that pass the hard rejection rules
    valid_candidates = [c for c in candidates if c[2]['valid']]
    yolo_best = max(valid_candidates, key=lambda c: c[2]['score'], default=None)

    if yolo_best and yolo_best[2]['score'] >= 0.45:
        quad, mask, scoring, cls_name = yolo_best
        method = 'yolo'
    else:
        # ── Path B: OpenCV-only fallback ──
        quad = _opencv_fallback_quad(img_rgb, debug=debug)
        if quad is None:
            if debug:
                print('  no card-like quadrilateral found via YOLO or OpenCV fallback')
            return None
        scoring = _card_likeness_score(quad, int(cv2.contourArea(quad.astype(np.int32))), img_area)
        mask = None
        cls_name = None
        method = 'opencv-fallback'
        if not scoring['valid']:
            if debug:
                print(f'  fallback returned invalid quad (sanity guard hit): {scoring}')
            return None

    if debug:
        print(f'  method={method}  class={cls_name}  score={scoring["score"]}  scoring={scoring}')

    ordered = _order_quad(quad)
    rectified = _perspective_warp(img_rgb, ordered, *canonical_size)
    aa_crop, aa_bbox = _axis_aligned_crop(img_rgb, ordered)

    return {
        'rectified':      rectified,
        'axis_aligned':   aa_crop,
        'bbox':           aa_bbox,
        'quad':           ordered,
        'method':         method,
        'detected_class': cls_name,
        'score':          scoring['score'],
        'scoring':        scoring,
        'mask':           mask,
    }


def visualize_yolo_crop(image, model_path: str = 'yolov8n-seg.pt',
                         figsize: tuple = (16, 6)) -> dict | None:
    """Side-by-side: original + overlay, axis-aligned crop, rectified card."""
    if isinstance(image, (str, _Path)):
        img_rgb = _load_rgb(str(image))
        original_path = str(image)
    else:
        img_rgb = image
        original_path = '<ndarray>'

    if img_rgb is None:
        print('  cannot load image'); return None

    result = yolo_crop_card(img_rgb, model_path=model_path, debug=True)
    if result is None:
        print('  no card detected')
        return None

    fig, axes = plt.subplots(1, 3, figsize=figsize, facecolor='#0d1117')
    for ax in axes: ax.set_facecolor('#0d1117'); ax.axis('off')

    # Panel 1: original with quad overlay
    axes[0].imshow(img_rgb)
    quad = result['quad']
    poly = mpatches.Polygon(quad, closed=True, fill=False,
                             edgecolor='#5fa8ff', linewidth=2.5)
    axes[0].add_patch(poly)
    # Numbered corner dots (TL=green, TR=blue, BR=red, BL=orange)
    corner_colors = ['#34d399', '#60a5fa', '#f87171', '#fb923c']
    corner_labels = ['TL', 'TR', 'BR', 'BL']
    for (x, y), c, lbl in zip(quad, corner_colors, corner_labels):
        axes[0].plot(x, y, 'o', color=c, markersize=10, markeredgecolor='white')
        axes[0].annotate(lbl, (x, y), color='white', fontsize=8,
                          xytext=(8, 8), textcoords='offset points', fontweight='bold')
    axes[0].set_title(
        f"Original + quad  ({result['method']}"
        + (f", class={result['detected_class']}" if result['detected_class'] else '')
        + f", score={result['score']:.2f})",
        color='white', fontsize=10,
    )

    # Panel 2: axis-aligned crop
    axes[1].imshow(result['axis_aligned'])
    aab = result['bbox']
    axes[1].set_title(
        f"Axis-aligned crop  ({aab['width']}×{aab['height']})",
        color='white', fontsize=10,
    )

    # Panel 3: perspective-rectified
    axes[2].imshow(result['rectified'])
    rh, rw = result['rectified'].shape[:2]
    axes[2].set_title(
        f"Rectified  ({rw}×{rh}, aspect {rw/rh:.3f})",
        color='white', fontsize=10,
    )

    plt.tight_layout()
    plt.show()
    return result


print('✅ YOLO card crop ready')
print('   • yolo_crop_card(img)         — returns dict with rectified + axis_aligned crops')
print('   • visualize_yolo_crop(img)    — side-by-side: original/quad → axis-aligned → rectified')
print('   • Fallback: OpenCV-only Canny + contour quad when YOLO finds nothing')


---
## Single Card Test

Paste eBay image URLs (s-l1600) **or** supply local file paths.  
Set `show_prompt=True` to print the exact prompt sent to Claude — useful when debugging prompt changes.

In [ ]:
# ── Configure your test card here ────────────────────────────────────────────

# Option A: eBay image URLs  (replace with real URLs)
TEST_IMAGE_URLS = [
    'https://i.ebayimg.com/images/g/REPLACE_ME/s-l1600.jpg',
    # add more URLs for multi-image tests
]

# Option B: Local files  (comment out TEST_IMAGE_URLS above and use this)
# TEST_LOCAL_PATHS = ['image0_front.jpeg', 'image0_back.jpeg']
TEST_LOCAL_PATHS = None

TEST_TITLE = 'Charizard ex 199/165 Scarlet & Violet 151 — PSA?'
TEST_PRICE = 480.0

SHOW_PROMPT = False   # True → print the full prompt sent to Claude
SHOW_RAW    = False   # True → print Claude's raw string before JSON parse

In [ ]:
# Show thumbnails first
if TEST_LOCAL_PATHS:
    show_images(local_paths=TEST_LOCAL_PATHS)
elif 'REPLACE_ME' not in TEST_IMAGE_URLS[0]:
    show_images(image_urls=TEST_IMAGE_URLS)

# Run grading
print('\nRunning grading pipeline...')
result = grade_with_claude(
    image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
    local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
    title=TEST_TITLE,
    show_prompt=SHOW_PROMPT,
    show_raw=SHOW_RAW,
)

print_result(result, label=TEST_TITLE)

In [ ]:
plot_result(result, label=TEST_TITLE)

# Panel-style centering inspection — same view as Chrome extension
first_image = (TEST_LOCAL_PATHS[0] if TEST_LOCAL_PATHS
               else TEST_IMAGE_URLS[0] if TEST_IMAGE_URLS
               else None)
if first_image:
    plot_centering_inspection_from_result(result, first_image)


In [ ]:
# Inspect the raw parsed JSON (excluding base64 image data)
display_json = {k: v for k, v in result.items() if not k.startswith('_')}
print(json.dumps(display_json, indent=2, ensure_ascii=False))

---
## Consistency Test — Same Card, N Runs

Run the same card multiple times to measure how consistent Claude's output is across calls.
Useful for understanding grade range variance before committing to a prompt.

In [ ]:
N_RUNS = 3   # number of times to repeat the same card

repeat_results = []
for i in tqdm(range(N_RUNS), desc='Consistency runs'):
    r = grade_with_claude(
        image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
        local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
        title=TEST_TITLE,
    )
    r['_run'] = i + 1
    repeat_results.append(r)

# Summary table
rows = []
for r in repeat_results:
    grade = r.get('grade_estimate', {})
    dist  = grade.get('distribution', {})
    gd    = r.get('grading_decision', {})
    rows.append({
        'Run':        r['_run'],
        'Grade Range': grade.get('grade_range', '?'),
        'Confidence':  grade.get('confidence', '?'),
        'Gradable':    gd.get('gradable_candidate', '?'),
        'P(PSA10)':   f"{dist.get('10',0)*100:.1f}%",
        'P(PSA9)':    f"{dist.get('9',0)*100:.1f}%",
        'P(PSA8)':    f"{dist.get('8',0)*100:.1f}%",
        'P(PSA7)':    f"{dist.get('7',0)*100:.1f}%",
        'In$':        r.get('_usage', {}).get('input_tokens', 0),
        'Out$':       r.get('_usage', {}).get('output_tokens', 0),
    })

df_repeat = pd.DataFrame(rows)
print(df_repeat.to_string(index=False))

In [ ]:
# Overlay distributions from all runs
fig, ax = plt.subplots(figsize=(12, 5))

grades = list(range(1, 11))
x = np.arange(len(grades))
width = 0.8 / N_RUNS
cmap = plt.cm.get_cmap('tab10', N_RUNS)

for idx, r in enumerate(repeat_results):
    dist  = r.get('grade_estimate', {}).get('distribution', {})
    probs = [dist.get(str(g), 0) * 100 for g in grades]
    offset = (idx - N_RUNS / 2 + 0.5) * width
    ax.bar(x + offset, probs, width=width * 0.9,
           label=f'Run {idx+1}', color=cmap(idx), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'PSA {g}' for g in grades])
ax.set_ylabel('Probability (%)')
ax.set_title(f'Grade Distribution Consistency — {N_RUNS} runs', color='white')
ax.legend()
ax.spines['bottom'].set_color('#30363d')
ax.spines['left'].set_color('#30363d')
plt.tight_layout()
plt.show()

---
## Batch Test — Multiple Cards

Define a list of test cases below (name, URLs or local paths, title, known grade if available) and run them all in one shot.

In [ ]:
# ── Define test cases ─────────────────────────────────────────────────────────
# Each dict keys:
#   name         : short label for display
#   image_urls   : list of eBay image URLs  (or use local_paths)
#   local_paths  : list of local image paths (comment out image_urls key)
#   title        : listing title (used as identity hint only)
#   price        : listing price in USD
#   known_grade  : actual PSA grade if you have it — used for accuracy calc (optional)

BATCH_CASES = [
    {
        'name':       'Card A',
        'local_paths': ['image0_front.jpeg', 'image0_back.jpeg'],
        'title':       'Card A — test',
        'price':       50.0,
        'known_grade': None,
    },
    {
        'name':       'Card B',
        'local_paths': ['image1_front.jpeg', 'image1_back.jpeg'],
        'title':       'Card B — test',
        'price':       30.0,
        'known_grade': None,
    },
    # Add more cases — paste eBay URLs or local paths
]

In [ ]:
batch_results = []

for i, tc in enumerate(BATCH_CASES):
    print(f"\n[{i+1}/{len(BATCH_CASES)}] {tc['name']}...")
    try:
        r = grade_with_claude(
            image_urls=tc.get('image_urls'),
            local_paths=tc.get('local_paths'),
            title=tc.get('title', ''),
        )
        r['_test_name']   = tc['name']
        r['_price']       = tc.get('price', 0)
        r['_known_grade'] = tc.get('known_grade')
        batch_results.append(r)
        print_result(r, label=tc['name'])
    except Exception as exc:
        print(f'  ❌ Error: {exc}')
        batch_results.append({
            '_test_name': tc['name'],
            '_error': str(exc)
        })

print(f'\n✅ Batch complete: {sum(1 for r in batch_results if "_error" not in r)}/{len(BATCH_CASES)} succeeded')

In [ ]:
# Comparison table
rows = []
for r in batch_results:
    if '_error' in r:
        rows.append({'Name': r['_test_name'], 'Error': r['_error']})
        continue
    grade  = r.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    card   = r.get('card_identity', {})
    gd     = r.get('grading_decision', {})
    iq     = r.get('image_quality', {})
    cv     = r.get('_cv') or {}
    front  = r.get('front_analysis', {})
    back   = r.get('back_analysis', {})
    known  = r.get('_known_grade')

    # Compute expected PSA from distribution
    ev = sum(int(g) * p for g, p in dist.items())

    rows.append({
        'Name':         r.get('_test_name', '?'),
        'Card':         card.get('name') or '?',
        'Mode':         r.get('analysis_mode', '?'),
        'Range':        grade.get('grade_range', '?'),
        'Conf':         grade.get('confidence', '?'),
        'Limit':        grade.get('limiting_factor') or '—',
        'E[PSA]':       f'{ev:.1f}',
        'Known':        known or '—',
        'Gradable':     gd.get('gradable_candidate', '?'),
        'Caveats':      len(gd.get('caveats', [])),
        'F.assess':     '✓' if front.get('assessable') else '✗',
        'B.assess':     '✓' if back.get('assessable') else '✗',
        'IQ':           iq.get('status', '?'),
        'Blur':         f"{cv.get('blur_score', '?')}",
        'Glare%':       f"{cv.get('glare_fraction', 0)*100:.1f}",
        'Issues':       sum(len(r.get('issues', {}).get(c, [])) for c in ['centering','corners','edges','surface','other']),
        'In tok':       r.get('_usage', {}).get('input_tokens', 0),
        'Out tok':      r.get('_usage', {}).get('output_tokens', 0),
    })

df_batch = pd.DataFrame(rows)
print(df_batch.to_string(index=False))

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results to plot')
else:
    n = len(valid)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, valid):
        dist   = r.get('grade_estimate', {}).get('distribution', {})
        probs  = [dist.get(str(g), 0) * 100 for g in range(1, 11)]
        colors = [GRADE_COLORS[g] for g in range(1, 11)]
        bars   = ax.bar(range(1, 11), probs, color=colors, width=0.7, edgecolor='#30363d')
        gr     = r.get('grade_estimate', {}).get('grade_range', '?')
        ax.set_title(f"{r.get('_test_name','?')}\n{gr}", color='white', fontsize=10)
        ax.set_xticks(range(1, 11))
        ax.set_xlabel('PSA Grade')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')
        for bar, prob in zip(bars, probs):
            if prob > 5:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,
                        f'{prob:.0f}%', ha='center', va='bottom',
                        color='white', fontsize=8)

    axes[0].set_ylabel('Probability (%)')
    plt.suptitle('Batch Grade Distributions', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Accuracy Analysis

If you supplied `known_grade` for any test cases, this section measures how well Claude's expected-value estimate matches the actual PSA grade.

In [ ]:
labelled = [
    r for r in batch_results
    if '_error' not in r and r.get('_known_grade') is not None
]

if not labelled:
    print('No labelled test cases (set known_grade in BATCH_CASES to enable accuracy analysis)')
else:
    errors, abs_errors = [], []
    for r in labelled:
        dist  = r.get('grade_estimate', {}).get('distribution', {})
        ev    = sum(int(g) * p for g, p in dist.items())
        known = float(r['_known_grade'])
        err   = ev - known
        errors.append(err)
        abs_errors.append(abs(err))
        print(f"  {r.get('_test_name','?'):20s}  E[PSA]={ev:.1f}  Actual={known:.0f}  Error={err:+.1f}")

    print(f'\n  MAE:  {np.mean(abs_errors):.2f} PSA grades')
    print(f'  Bias: {np.mean(errors):+.2f} (positive = overestimates grade)')
    print(f'  Within ±1 grade: {sum(1 for e in abs_errors if e <= 1)}/{len(abs_errors)}')
    print(f'  Within ±2 grade: {sum(1 for e in abs_errors if e <= 2)}/{len(abs_errors)}')

---
## Issues Frequency Analysis

Aggregates which issue categories appear most often across your batch — useful for spotting systematic bias in the prompt.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']
    counts = {c: 0 for c in cats}
    total_issues = 0
    for r in valid:
        for cat in cats:
            n = len(r.get('issues', {}).get(cat, []))
            counts[cat] += n
            total_issues += n

    fig, ax = plt.subplots(figsize=(9, 4))
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']
    bars = ax.bar(cat_labels, [counts[c] for c in cats],
                  color=bar_colors, edgecolor='#30363d')
    for bar, cat in zip(bars, cats):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                str(counts[cat]),
                ha='center', va='bottom', color='white', fontsize=11)
    ax.set_ylabel('Total issues reported')
    ax.set_title(f'Issue category frequency — {len(valid)} card(s), {total_issues} total issues',
                 color='white')
    ax.spines['bottom'].set_color('#30363d')
    ax.spines['left'].set_color('#30363d')
    plt.tight_layout()
    plt.show()

    print('\nAll reported issues:')
    for cat in cats:
        all_items = []
        for r in valid:
            all_items.extend(r.get('issues', {}).get(cat, []))
        if all_items:
            print(f'\n  {cat.upper()}')
            for item in all_items:
                print(f'    \u2022 {item}')

---
## Side Analysis Summary

Per-side (front vs back) issue frequency breakdown and assessability overview across the batch.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']

    # ── Assessability summary ─────────────────────────────────────
    front_assess = sum(1 for r in valid if r.get('front_analysis', {}).get('assessable', True))
    back_assess  = sum(1 for r in valid if r.get('back_analysis',  {}).get('assessable', True))
    front_only   = sum(1 for r in valid if r.get('analysis_mode') == 'front_only')
    front_back   = sum(1 for r in valid if r.get('analysis_mode') == 'front_back')
    print(f'Cards analysed: {len(valid)}')
    print(f'  front_back mode : {front_back}')
    print(f'  front_only mode : {front_only}')
    print(f'  Front assessable: {front_assess}/{len(valid)}')
    print(f'  Back  assessable: {back_assess}/{len(valid)}')

    # ── Per-side issue counts ─────────────────────────────────────
    front_counts = {c: 0 for c in cats}
    back_counts  = {c: 0 for c in cats}
    for r in valid:
        for cat in cats:
            front_counts[cat] += len(r.get('front_analysis', {}).get('issues', {}).get(cat, []))
            back_counts[cat]  += len(r.get('back_analysis',  {}).get('issues', {}).get(cat, []))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']

    for ax, counts, title in [
        (axes[0], front_counts, 'Front Issues'),
        (axes[1], back_counts,  'Back Issues'),
    ]:
        bars = ax.bar(cat_labels, [counts[c] for c in cats],
                      color=bar_colors, edgecolor='#30363d')
        for bar, cat in zip(bars, cats):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    str(counts[cat]),
                    ha='center', va='bottom', color='white', fontsize=11)
        ax.set_title(title, color='white')
        ax.set_ylabel('Issue count')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')

    plt.suptitle('Front vs Back Issue Breakdown', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Caveats summary ───────────────────────────────────────────
    caveats_total = sum(len(r.get('grading_decision', {}).get('caveats', [])) for r in valid)
    if caveats_total:
        print(f'\n⛔  {caveats_total} caveat(s) across {len(valid)} card(s):')
        for r in valid:
            for c in r.get('grading_decision', {}).get('caveats', []):
                print(f'  [{r.get("_test_name","?")}] {c}')
    else:
        print('\n✓  No caveats — all cards had back images available')